# Stage 1 — Preprocessing: ADNI4 Raw Data Inventory, Linkage, and Source Dataset Export

**Module Role (`src/preprocessing`):** Audits raw ADNI4 clinical (`DXSUM`), informant cognition (`ECOG12PT`), demographics (`PTDEMOG`), and speech (`STORYTELLER`) files, performs diagnosis/visit linkage, and exports only the source datasets used downstream to `data/processed/`:
- `dataset1_storyteller.csv` (Storyteller assessments + labels)
- `dataset2_ecog.csv` (ECog-12 PT questionnaire records + labels)
- `dataset4_multimodal_cn_vs_noncn.csv` (separate all-pairs sensitivity extension)

**Primary 30-Day Cohort:** The unused legacy `dataset3_multimodal.csv` export has been removed. The canonical 77-assessment / 71-participant primary 30-day matched cohort is constructed and saved once in `03_train_temporal_and_ablation.ipynb` (`data/processed/primary_multimodal_30day_cohort.csv`).


# ADNI4 Raw Data Inventory and Cohort Preparation

This notebook organizes the supplied Python script into readable sections. It inventories the configured raw CSV files, audits visit and diagnosis linkage, reviews Storyteller and ECOG data quality, and constructs and exports the three prepared research datasets.

Run the code cells from top to bottom; later sections use dataframes and variables created earlier. The source code, including its comments, is preserved unchanged.


## Raw-file inventory and project setup

Set project paths, list configured CSVs, inspect files, and save the initial column inventory.


In [39]:
# Generated from: 01_list_raw_files.ipynb
# Converted at: 2026-11-22T16:12:08.273Z
# Next step (optional): refactor into modules & generate tests with RunCell
# Quick start: pip install runcell

from pathlib import Path
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# --------------------------------------------------
# 1. FOLDER PATHS
# --------------------------------------------------

PROJECT_DIR = Path("DATASET INSPECTION").resolve().parent.parent
RAW_DIR = PROJECT_DIR / "data"/"raw"
RESULTS_DIR = PROJECT_DIR / "data"/"processed"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. DATASET FILES
# --------------------------------------------------

DATASETS = {
    "DXSUM": "All_Subjects_DXSUM_06Nov2026.csv",
    "ECOG12PT": "All_Subjects_ECOG12PT_06Nov2026.csv",
    "PTDEMOG": "All_Subjects_PTDEMOG_06Nov2026.csv",
    "DATADIC": "DATADIC_06Nov2026.csv",
    "STORYTELLER": "RMT_STORYTELLER_06Nov2026.csv"
}

# --------------------------------------------------
# 3. HELPER FUNCTIONS
# --------------------------------------------------

def load_csv(path):
    """Load CSV with common encoding fallbacks."""

    try:
        return pd.read_csv(path, low_memory=False, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="cp1252")


def find_candidate_columns(columns):
    """Identify columns potentially useful for linkage and analysis."""

    keywords = [
        "PTID", "RID", "VISCODE", "VISIT",
        "DATE", "EXAMDATE", "DX", "DIAG",
        "ECOG", "STORY", "SCREENER"
    ]

    candidates = []

    for col in columns:
        upper = str(col).upper()

        if any(keyword in upper for keyword in keywords):
            candidates.append(col)

    return candidates


# --------------------------------------------------
# 4. INSPECT ALL DATASETS
# --------------------------------------------------

report_path = RESULTS_DIR / "dataset_audit_report.txt"
inventory_path = RESULTS_DIR / "column_inventory.csv"

inventory_rows = []

with open(report_path, "w", encoding="utf-8") as report:

    report.write("=" * 90 + "\n")
    report.write("ADNI4 DATASET AUDIT - 06 NOVEMBER 2026\n")
    report.write("=" * 90 + "\n")

    for dataset_name, filename in DATASETS.items():

        file_path = RAW_DIR / filename

        report.write("\n\n" + "=" * 90 + "\n")
        report.write(f"DATASET: {dataset_name}\n")
        report.write(f"FILE: {filename}\n")
        report.write("=" * 90 + "\n")

        if not file_path.exists():
            report.write("ERROR: File not found.\n")
            continue

        df = load_csv(file_path)

        # Basic information
        rows, cols = df.shape

        report.write(f"\nRows: {rows:,}\n")
        report.write(f"Columns: {cols:,}\n")
        report.write(f"Exact duplicate rows: {df.duplicated().sum():,}\n")

        report.write("\n--- COLUMN INFORMATION ---\n")

        for col in df.columns:

            missing = df[col].isna().sum()
            missing_pct = (missing / len(df) * 100) if len(df) else 0
            unique = df[col].nunique(dropna=True)

            report.write(
                f"\nColumn: {col}\n"
                f"  Data type: {df[col].dtype}\n"
                f"  Missing: {missing:,} ({missing_pct:.2f}%)\n"
                f"  Unique values: {unique:,}\n"
            )

            inventory_rows.append({
                "dataset": dataset_name,
                "column": col,
                "dtype": str(df[col].dtype),
                "missing_count": missing,
                "missing_percentage": round(missing_pct, 2),
                "unique_values": unique
            })

        # Candidate linkage and research columns
        candidates = find_candidate_columns(df.columns)

        report.write("\n\n--- IMPORTANT CANDIDATE COLUMNS ---\n")

        if not candidates:
            report.write("No candidate columns identified.\n")

        for col in candidates:

            report.write(f"\n{col}\n")

            examples = (
                df[col]
                .dropna()
                .astype(str)
                .drop_duplicates()
                .head(10)
                .tolist()
            )

            report.write(f"  Example values: {examples}\n")

        # Possible participant and visit duplicates
        id_columns = [
            c for c in df.columns
            if str(c).upper() in ["PTID", "SUBJID"]
        ]

        visit_columns = [
            c for c in df.columns
            if str(c).upper() in ["VISCODE", "VISCODE2"]
        ]

        if id_columns and visit_columns:

            id_col = id_columns[0]
            visit_col = visit_columns[0]

            duplicate_pairs = df.duplicated(
                subset=[id_col, visit_col],
                keep=False
            ).sum()

            report.write(
                f"\nDuplicate participant-visit rows "
                f"({id_col} + {visit_col}): {duplicate_pairs:,}\n"
            )

        # Show a small structural preview
        report.write("\n--- FIRST 5 ROWS (STRUCTURAL PREVIEW) ---\n")
        report.write(df.head(5).to_string(index=False))
        report.write("\n")

        print(
            f"{dataset_name:15} | "
            f"Rows: {rows:8,} | "
            f"Columns: {cols:4,} | "
            f"Exact duplicates: {df.duplicated().sum():,}"
        )

# --------------------------------------------------
# 5. SAVE COLUMN INVENTORY
# --------------------------------------------------





print("\n" + "=" * 70)
print("AUDIT COMPLETED")
print("=" * 70)
print(f"Full report: {report_path}")
print(f"Column inventory: {inventory_path}")



# ============================================================


DXSUM           | Rows:   16,614 | Columns:   41 | Exact duplicates: 0
ECOG12PT        | Rows:    1,680 | Columns:   31 | Exact duplicates: 0
PTDEMOG         | Rows:    6,244 | Columns:   84 | Exact duplicates: 0
DATADIC         | Rows:   79,317 | Columns:   13 | Exact duplicates: 1
STORYTELLER     | Rows:   15,878 | Columns:   72 | Exact duplicates: 0

AUDIT COMPLETED
Full report: C:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\data\processed\dataset_audit_report.txt
Column inventory: C:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\data\processed\column_inventory.csv


## Dataset details

Review dataset shapes, data types, missingness, example rows, and candidate linkage fields.


In [40]:
# STEP 2: DISPLAY DATASET DETAILS
# Reuses the variables and folders from the previous cell
# ============================================================

from IPython.display import display

# Load datasets into memory for subsequent analysis
data = {}

for name, filename in DATASETS.items():
    data[name] = load_csv(RAW_DIR / filename)

print("=" * 85)
print("ADNI4 DATASET DETAILS")
print("=" * 85)

for name, df in data.items():

    print(f"\n{'=' * 85}")
    print(f"DATASET: {name}")
    print(f"{'=' * 85}")

    print(f"Shape: {df.shape}")
    print(f"Duplicate rows: {df.duplicated().sum()}")

    print("\nCOLUMN NAMES AND DATA TYPES:")
    display(
        pd.DataFrame({
            "Column": df.columns,
            "Data Type": df.dtypes.astype(str).values,
            "Missing": df.isna().sum().values,
            "Missing %": (
                df.isna().mean().values * 100
            ).round(2),
            "Unique Values": df.nunique().values
        })
    )

    print("\nFIRST 5 ROWS:")
    display(df.head())

    print("\nIMPORTANT CANDIDATE COLUMNS:")
    candidates = find_candidate_columns(df.columns)

    for col in candidates:
        examples = (
            df[col]
            .dropna()
            .astype(str)
            .unique()[:8]
        )

        print(f"{col}: {list(examples)}")



# ============================================================


ADNI4 DATASET DETAILS

DATASET: DXSUM
Shape: (16614, 41)
Duplicate rows: 0

COLUMN NAMES AND DATA TYPES:


,Column,Data Type,Missing,Missing %,Unique Values
0,PHASE,str,0,0.00,6
1,PTID,str,0,0.00,3799
2,RID,int64,0,0.00,3799
3,VISCODE,str,0,0.00,33
4,VISCODE2,str,11,0.07,45
5,EXAMDATE,str,110,0.66,4557
6,DIAGNOSIS,float64,46,0.28,4
7,DXNORM,float64,12746,76.72,2
8,DXNODEP,float64,12746,76.72,2
9,DXMCI,float64,12746,76.72,2



FIRST 5 ROWS:


,PHASE,PTID,RID,VISCODE,VISCODE2,EXAMDATE,DIAGNOSIS,DXNORM,DXNODEP,DXMCI,...,DXODES,DXCONFID,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp
0,ADNI1,011_S_0002,2,bl,bl,2005-09-29,1.0,1.0,-4.0,-4.0,...,-4.0,4.0,2,107,2005-10-01,NaN,NaN,NaN,NaN,2005-10-01 00:00:00
1,ADNI1,011_S_0003,3,bl,bl,2005-09-30,3.0,-4.0,-4.0,-4.0,...,-4.0,3.0,4,107,2005-10-01,NaN,NaN,NaN,NaN,2005-10-01 00:00:00
2,ADNI1,011_S_0005,5,bl,bl,2005-09-30,1.0,1.0,-4.0,-4.0,...,-4.0,4.0,6,107,2005-10-01,NaN,NaN,NaN,NaN,2005-10-01 00:00:00
3,ADNI1,011_S_0008,8,bl,bl,2005-09-30,1.0,1.0,-4.0,-4.0,...,-4.0,3.0,8,107,2005-10-01,NaN,NaN,NaN,NaN,2005-10-01 00:00:00
4,ADNI1,022_S_0007,7,bl,bl,2005-10-06,3.0,-4.0,-4.0,-4.0,...,-4.0,4.0,10,10,2005-10-06,NaN,NaN,NaN,NaN,2005-10-06 00:00:00



IMPORTANT CANDIDATE COLUMNS:
PTID: ['011_S_0002', '011_S_0003', '011_S_0005', '011_S_0008', '022_S_0007', '100_S_0015', '023_S_0030', '023_S_0031']
RID: ['2', '3', '5', '8', '7', '15', '30', '31']
VISCODE: ['bl', 'm06', 'm12', 'uns1', 'm36', 'm18', 'm24', 'm48']
VISCODE2: ['bl', 'm06', 'm12', 'uns1', 'm36', 'm18', 'm24', 'm48']
EXAMDATE: ['2005-09-29', '2005-09-30', '2005-10-06', '2005-10-18', '2005-10-20', '2005-10-24', '2005-10-25', '2005-11-01']
DIAGNOSIS: ['1.0', '3.0', '2.0', '10.0']
DXNORM: ['1.0', '-4.0']
DXNODEP: ['-4.0', '1.0']
DXMCI: ['-4.0', '1.0']
DXMDES: ['-4', '1', '1|2', '2']
DXMPTR1: ['-4.0', '1.0', '0.0']
DXMPTR2: ['-4.0', '1.0', '0.0']
DXMPTR3: ['-4.0', '1.0', '2.0', '0.0']
DXMPTR4: ['-4.0', '1.0', '2.0', '0.0']
DXMPTR5: ['-4.0', '1.0', '0.0']
DXMPTR6: ['-4.0', '1.0', '0.0']
DXMDUE: ['-4.0', '1.0', '2.0']
DXMOTHET: ['-4', '9', '14', '1', '9|14', '7|9', '7', '4']
DXDSEV: ['-4.0', '2.0', '1.0', '3.0']
DXDDUE: ['1.0', '2.0']
DXAD: ['-4.0', '1.0']
DXAPP: ['-4.0', '1.0', 

,Column,Data Type,Missing,Missing %,Unique Values
0,PHASE,str,0,0.00,1
1,PTID,str,0,0.00,1039
2,RID,int64,0,0.00,1039
3,VISCODE,str,0,0.00,5
4,VISCODE2,str,2,0.12,40
5,COLDATE,str,4,0.24,618
6,STAFFASST,float64,29,1.73,4
7,VALIDITY,float64,36,2.14,4
8,SOURCE,float64,29,1.73,2
9,CONCERN,float64,38,2.26,2


... (11 outputs left out)

,Column,Data Type,Missing,Missing %,Unique Values
0,RMT_PHASE,str,0,0.00,1
1,ADNIOnlineID,str,0,0.00,8798
2,PTID,str,14206,89.47,539
3,RMT_Timepoint,str,0,0.00,6
4,EXAMDATE,str,0,0.00,974
...,...,...,...,...,...
67,STORY1DEL_SPEECH_RATE,float64,302,1.90,9281
68,STORY1DEL_ARTICULATION_RATE,float64,302,1.90,7629
69,STORY1DEL_NUM_PAUSES,float64,12,0.08,37
70,STORY1DEL_MEAN_PAUSE_LENGTH,float64,368,2.32,2198



FIRST 5 ROWS:


,RMT_PHASE,ADNIOnlineID,PTID,RMT_Timepoint,EXAMDATE,SCREENER_SCORE,SCREENER_RECOMMEND,QC_ATTEMPTS,QC_PASSED,USER_AGENT,...,STORY1DEL_MEAN_LETTER_PER_WORD,STORY1DEL_GMATCH,STORY1DEL_GMATCH_IMM_DEL,STORY1DEL_SPEECH_DURATION,STORY1DEL_NONSPEECH_PROPORTION,STORY1DEL_SPEECH_RATE,STORY1DEL_ARTICULATION_RATE,STORY1DEL_NUM_PAUSES,STORY1DEL_MEAN_PAUSE_LENGTH,update_stamp
0,ADNI4,ADNI4-PYP-9P9,NaN,m00,2023-06-21,57,0,1,1,"Mozilla/5.0 (iPad; CPU OS 15_6_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Mobile/19G82 [FBAN/FBIOS;FBDV/iPad6,11;FBMD/iPad;FBSN/iPadOS;FBSV/15.6.1;FBSS/2;FBID/tablet;FBLC/en_US;FBOP/5]",...,4.07692,54.302493,97.728266,21.3,0.262976,1.799308,2.441315,9.0,1.122222,2026-02-04 08:17:07
1,ADNI4,ADNI4-PYP-9P9,NaN,m06,2024-01-12,57,0,2,1,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.6.1 Safari/605.1.15",...,3.67857,48.323083,81.881903,18.6,0.424149,1.733746,3.010753,10.0,1.620000,2026-02-04 08:17:07
2,ADNI4,ADNI4-PGP-PNH,NaN,m00,2023-06-28,72,0,2,1,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",...,4.21505,68.286296,97.707617,33.7,0.174020,2.279412,2.759644,13.0,0.600000,2026-02-04 08:17:07
3,ADNI4,ADNI4-PGP-PNH,NaN,m06,2024-01-02,64,0,1,1,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36",...,4.31707,46.838374,55.249441,29.7,0.172702,2.284123,2.760943,9.0,0.711111,2026-02-04 08:17:07
4,ADNI4,ADNI4-PGP-PNH,NaN,m12,2024-07-03,71,0,1,1,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",...,3.88333,70.032840,84.026648,42.2,0.023148,2.777778,2.843602,2.0,0.700000,2026-02-04 08:17:07



IMPORTANT CANDIDATE COLUMNS:
PTID: ['941_S_6254', '941_S_6454', '022_S_10043', '033_S_10005', '941_S_6570', '033_S_7100', '033_S_10009', '941_S_6058']
EXAMDATE: ['2023-06-21', '2024-01-12', '2023-06-28', '2024-01-02', '2024-07-03', '2025-01-07', '2023-07-05', '2024-01-10']
SCREENER_SCORE: ['57', '72', '64', '71', '84', '73', '68', '75']
SCREENER_RECOMMEND: ['0', '1']
STORY1IMM_LANGUAGE: ['en-US', 'es-US']
STORY1IMM_PLAYBACK_COUNT: ['1.0', '0.0', '2.0', '3.0', '4.0', '5.0']
STORY1IMM_AUDIO_LENGTH: ['30.6', '35.46', '45.38', '42.06', '63.84', '68.46', '27.28', '37.74']
STORY1IMM_NUM_PRONOUNS: ['7.0', '14.0', '16.0', '26.0', '19.0', '15.0', '18.0', '13.0']
STORY1IMM_NUM_NOUNS: ['7.0', '17.0', '15.0', '19.0', '21.0', '23.0', '12.0', '13.0']
STORY1IMM_NUM_PROPER_NOUNS: ['2.0', '5.0', '4.0', '1.0', '3.0', '8.0', '0.0', '7.0']
STORY1IMM_NUM_VERBS: ['5.0', '8.0', '14.0', '13.0', '18.0', '19.0', '11.0', '9.0']
STORY1IMM_NUM_ADVERBS: ['0.0', '2.0', '7.0', '5.0', '9.0', '6.0', '3.0', '8.0']
STOR

## ADNI4 diagnosis and visit linkage

Inspect ADNI4 diagnosis coverage and compare participant and visit identifiers across sources.


In [41]:
# STEP 3: ADNI4 DIAGNOSIS, VISIT AND PARTICIPANT MATCHING
# ============================================================

dx = data["DXSUM"].copy()
ecog = data["ECOG12PT"].copy()
story = data["STORYTELLER"].copy()
dic = data["DATADIC"].copy()

# ------------------------------------------------------------
# A. PHASE DISTRIBUTION IN DXSUM
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("A. DXSUM PHASE DISTRIBUTION")
print("=" * 80)

display(dx["PHASE"].value_counts(dropna=False).to_frame("Records"))

# Isolate ADNI4 diagnosis records
dx4 = dx[dx["PHASE"].str.upper() == "ADNI4"].copy()

print("ADNI4 DXSUM shape:", dx4.shape)
print("ADNI4 unique participants:", dx4["PTID"].nunique())

# ------------------------------------------------------------
# B. ADNI4 DIAGNOSIS DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("B. ADNI4 DIAGNOSIS DISTRIBUTION")
print("=" * 80)

print("DIAGNOSIS value counts:")
display(dx4["DIAGNOSIS"].value_counts(dropna=False).sort_index().to_frame("Records"))

print("\nDiagnosis by visit:")
display(
    pd.crosstab(
        dx4["VISCODE"],
        dx4["DIAGNOSIS"],
        dropna=False
    )
)

# ------------------------------------------------------------
# C. VISIT CODE DISTRIBUTIONS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("C. VISIT CODE DISTRIBUTIONS")
print("=" * 80)

print("\nECOG VISCODE:")
display(ecog["VISCODE"].value_counts(dropna=False).to_frame("Records"))

print("\nECOG VISCODE2:")
display(ecog["VISCODE2"].value_counts(dropna=False).head(30).to_frame("Records"))

print("\nADNI4 DXSUM VISCODE:")
display(dx4["VISCODE"].value_counts(dropna=False).to_frame("Records"))

print("\nADNI4 DXSUM VISCODE2:")
display(dx4["VISCODE2"].value_counts(dropna=False).head(30).to_frame("Records"))

# ------------------------------------------------------------
# D. PARTICIPANT OVERLAP
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("D. PARTICIPANT OVERLAP")
print("=" * 80)

ecog_ids = set(ecog["PTID"].dropna().unique())
dx_ids = set(dx4["PTID"].dropna().unique())
story_ids = set(story["PTID"].dropna().unique())

print("Unique ECOG participants:", len(ecog_ids))
print("Unique ADNI4 DXSUM participants:", len(dx_ids))
print("Unique Storyteller participants with PTID:", len(story_ids))

print("\nECOG ∩ ADNI4 DXSUM:", len(ecog_ids & dx_ids))
print("ECOG ∩ Storyteller:", len(ecog_ids & story_ids))
print("Storyteller ∩ ADNI4 DXSUM:", len(story_ids & dx_ids))

print("\nAll three datasets:")
print(len(ecog_ids & dx_ids & story_ids))

# ------------------------------------------------------------
# E. PARTICIPANT + VISIT PAIR OVERLAP
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("E. PARTICIPANT-VISIT MATCHING")
print("=" * 80)

ecog_pairs = set(
    zip(ecog["PTID"], ecog["VISCODE2"])
)

dx_pairs = set(
    zip(dx4["PTID"], dx4["VISCODE2"])
)

ecog_pairs.discard((None, None))
dx_pairs.discard((None, None))

exact_pairs = ecog_pairs & dx_pairs

print("Unique ECOG participant-visit pairs:", len(ecog_pairs))
print("Unique DXSUM participant-visit pairs:", len(dx_pairs))
print("Exact PTID + VISCODE2 matches:", len(exact_pairs))

# Compare after removing missing keys
ecog_valid = ecog.dropna(subset=["PTID", "VISCODE2"])
dx_valid = dx4.dropna(subset=["PTID", "VISCODE2"])

ecog_pair_df = ecog_valid[["PTID", "VISCODE2"]].drop_duplicates()
dx_pair_df = dx_valid[["PTID", "VISCODE2"]].drop_duplicates()

pair_check = ecog_pair_df.merge(
    dx_pair_df,
    on=["PTID", "VISCODE2"],
    how="inner"
)

print("Verified exact pair matches:", len(pair_check))

# ------------------------------------------------------------
# F. DATA DICTIONARY: RELEVANT DEFINITIONS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("F. DATA DICTIONARY DEFINITIONS")
print("=" * 80)

fields_to_check = [
    "DIAGNOSIS",
    "VISCODE",
    "VISCODE2",
    "EXAMDATE",
    "COLDATE",
    "STAFFASST",
    "VALIDITY",
    "SOURCE",
    "CONCERN",
    "MEMREF"
] + [f"ECOGPT{i}" for i in range(1, 13)]

relevant_dic = dic[
    dic["FLDNAME"].astype(str).str.upper().isin(fields_to_check)
].copy()

display(
    relevant_dic[
        [
            "PHASE", "CRFNAME", "TBLNAME", "FLDNAME",
            "TEXT", "CODE", "UNITS", "MAPPING_NOTES"
        ]
    ].drop_duplicates()
)

# ------------------------------------------------------------
# G. STORYTELLER PARTICIPANT ID AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("G. STORYTELLER IDENTIFIER AUDIT")
print("=" * 80)

print("Total Storyteller rows:", len(story))
print("Missing PTID:", story["PTID"].isna().sum())
print("Nonmissing PTID:", story["PTID"].notna().sum())
print("Unique PTID:", story["PTID"].nunique())

print("\nRMT_Timepoint distribution:")
display(story["RMT_Timepoint"].value_counts(dropna=False).to_frame("Records"))

print("\nStoryteller rows by PTID availability:")
display(
    story.assign(
        PTID_status=story["PTID"].notna().map({
            True: "PTID available",
            False: "PTID missing"
        })
    )["PTID_status"].value_counts().to_frame("Records")
)

print("\nADNI4 STORYTELLER AND ECOG PARTICIPANT OVERLAP:")
print("Matched participants:", len(story_ids & ecog_ids))

print("\nSTEP 3 COMPLETED")


# ------------------------------------------------------------
# E. PARTICIPANT + VISIT PAIR OVERLAP
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("E. PARTICIPANT-VISIT MATCHING")
print("=" * 80)

ecog_pair_df = (
    ecog.dropna(subset=["PTID", "VISCODE2"])
    [["PTID", "VISCODE2"]]
    .drop_duplicates()
)

dx_pair_df = (
    dx4.dropna(subset=["PTID", "VISCODE2"])
    [["PTID", "VISCODE2"]]
    .drop_duplicates()
)

pair_check = ecog_pair_df.merge(
    dx_pair_df,
    on=["PTID", "VISCODE2"],
    how="inner"
)

print("Unique ECOG participant-visit pairs:", len(ecog_pair_df))
print("Unique DXSUM participant-visit pairs:", len(dx_pair_df))
print("Exact PTID + VISCODE2 matches:", len(pair_check))

print("ECOG pairs without DXSUM match:",
      len(ecog_pair_df) - len(pair_check))

# ============================================================



A. DXSUM PHASE DISTRIBUTION


,Records
PHASE,
ADNI2,5671
ADNI1,3868
ADNI3,3587
ADNI4,2925
ADNIGO,475
TEAM,88


ADNI4 DXSUM shape: (2925, 41)
ADNI4 unique participants: 1273

B. ADNI4 DIAGNOSIS DISTRIBUTION
DIAGNOSIS value counts:


,Records
DIAGNOSIS,
1.0,1715
2.0,888
3.0,316
NaN,6



Diagnosis by visit:


DIAGNOSIS,1.0,2.0,3.0,NaN
VISCODE,,,,
4_bl,302,184,49,1
4_init,332,141,55,0
4_m12,455,228,94,1
4_m24,187,91,44,0
4_m36,10,10,0,0
4_sc,429,234,74,4


... (11 outputs left out)

,Records
RMT_Timepoint,
m00,8187
m06,4467
m12,2356
m18,679
m24,156
m30,33



Storyteller rows by PTID availability:


,Records
PTID_status,
PTID missing,14206
PTID available,1672



ADNI4 STORYTELLER AND ECOG PARTICIPANT OVERLAP:
Matched participants: 490

STEP 3 COMPLETED

E. PARTICIPANT-VISIT MATCHING
Unique ECOG participant-visit pairs: 1678
Unique DXSUM participant-visit pairs: 2921
Exact PTID + VISCODE2 matches: 1620
ECOG pairs without DXSUM match: 58


## Diagnosis quality in matched ECOG visits

Check duplicate diagnosis visits, attach DXSUM labels, and inspect diagnosis distributions.


In [42]:
# STEP 4: DIAGNOSIS QUALITY IN MATCHED ECOG VISITS
# ============================================================

# 1. Check duplicate participant-visit keys in DXSUM
dx_key_counts = (
    dx4.dropna(subset=["PTID", "VISCODE2"])
    .groupby(["PTID", "VISCODE2"])
    .size()
    .reset_index(name="DXSUM_records")
)

print("Participant-visit keys with multiple DXSUM records:",
      (dx_key_counts["DXSUM_records"] > 1).sum())

display(
    dx_key_counts[dx_key_counts["DXSUM_records"] > 1]
    .sort_values("DXSUM_records", ascending=False)
    .head(20)
)

# 2. Attach diagnosis information to matched ECOG visits
dx_visit_info = dx4[
    ["PTID", "VISCODE2", "DIAGNOSIS", "EXAMDATE"]
].copy()

matched_ecog_dx = ecog.merge(
    dx_visit_info,
    on=["PTID", "VISCODE2"],
    how="inner"
)

print("\nMatched ECOG-DXSUM rows:", len(matched_ecog_dx))

# 3. Diagnosis distribution in matched visits
print("\nDiagnosis distribution in matched ECOG visits:")
display(
    matched_ecog_dx["DIAGNOSIS"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Records")
)

# 4. Check whether merging created duplicate ECOG visits
print("\nECOG rows before matching:", len(ecog))
print("Matched rows after merging:", len(matched_ecog_dx))
print("Unique matched participant-visit pairs:",
      matched_ecog_dx[["PTID", "VISCODE2"]].drop_duplicates().shape[0])

# 5. Show dictionary definitions for diagnosis and ECOG coding
display(
    relevant_dic[
        relevant_dic["FLDNAME"].str.upper().isin(
            ["DIAGNOSIS"] + [f"ECOGPT{i}" for i in range(1, 13)]
        )
    ][
        ["PHASE", "CRFNAME", "TBLNAME", "FLDNAME",
         "TEXT", "CODE", "MAPPING_NOTES"]
    ].drop_duplicates()
)

# ============================================================


Participant-visit keys with multiple DXSUM records: 1


,PTID,VISCODE2,DXSUM_records
2283,129_S_6452,m90,2



Matched ECOG-DXSUM rows: 1622

Diagnosis distribution in matched ECOG visits:


,Records
DIAGNOSIS,
1.0,779
2.0,634
3.0,207
NaN,2



ECOG rows before matching: 1680
Matched rows after merging: 1622
Unique matched participant-visit pairs: 1622


,PHASE,CRFNAME,TBLNAME,FLDNAME,TEXT,CODE,MAPPING_NOTES
26272,ADNI2,Clinician Verification,CLIELG,DIAGNOSIS,Specify diagnostic category:,1=Cognitively Normal; 5=Significant Memory Concern;2=Early MCI; 3=Late MCI; 4=Alzheimer's Disease,NaN
28213,ADNI3,Diagnostic Summary,DXSUM,DIAGNOSIS,1. Which best describes the participant's current diagnosis?,1=CN;2=MCI;3=Dementia,NaN
28223,ADNI4,Diagnostic Summary,DXSUM,DIAGNOSIS,1. Which best describes the participant's current diagnosis?,1=CN; 2=MCI; 3=Dementia,NaN
28259,TEAM,Diagnostic Summary,DXSUM,DIAGNOSIS,1. Which best describes the participant's current diagnosis?,1=CN; 2=MCI; 3=Dementia; 10=TEAM_NODX,Assigned constant value 10 (TEAM_NODX) because TEAM does not collect a formal diagnosis assessment
28326,ADNI4,Everyday Cognition 12 Item - Participant Self-Report,ECOG12PT,ECOGPT1,1. Remembering where you have placed personal items or objects:,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28327,ADNI4,Everyday Cognition 12 Item - Participant Self-Report,ECOG12PT,ECOGPT10,10. Managing bill payments,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28328,ADNI4,Everyday Cognition 12 Item - Participant Self-Report,ECOG12PT,ECOGPT11,11. Doing two things at once,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28329,ADNI4,Everyday Cognition 12 Item - Participant Self-Report,ECOG12PT,ECOGPT12,12. Working on a particular task while talking to someone at the same time,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28330,ADNI4,Everyday Cognition 12 Item - Participant Self-Report,ECOG12PT,ECOGPT2,2. Remembering the current date or day of the week,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28331,ADNI4,Everyday Cognition 12 Item - Participant Self-Report,ECOG12PT,ECOGPT3,3. Communicating or expressing ideas in a conversation,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN


## Duplicate visits and ECOG response codes

Review duplicated visits and validate ECOG item response values.


In [43]:
# STEP 5: DUPLICATE VISITS AND ECOG RESPONSE CODE VALIDATION
# ============================================================

# ------------------------------------------------------------
# A. Inspect the duplicated DXSUM participant-visit
# ------------------------------------------------------------

print("\nA. DUPLICATED DXSUM RECORDS")

duplicate_dx = dx4[
    (dx4["PTID"] == "129_S_6452") &
    (dx4["VISCODE2"] == "m90")
]

display(duplicate_dx)

# Show all columns that differ between the two records
print("\nValues for key diagnosis and date fields:")
display(
    duplicate_dx[
        ["PTID", "RID", "VISCODE", "VISCODE2",
         "EXAMDATE", "DIAGNOSIS", "PHASE"]
    ]
)

# ------------------------------------------------------------
# B. Check whether ECOG itself has duplicate visit keys
# ------------------------------------------------------------

print("\nB. DUPLICATED ECOG PARTICIPANT-VISIT KEYS")

ecog_key_counts = (
    ecog.dropna(subset=["PTID", "VISCODE2"])
    .groupby(["PTID", "VISCODE2"])
    .size()
    .reset_index(name="ECOG_records")
)

ecog_duplicates = ecog_key_counts[
    ecog_key_counts["ECOG_records"] > 1
]

print("Number of duplicated ECOG keys:", len(ecog_duplicates))
display(ecog_duplicates)

# ------------------------------------------------------------
# C. Display complete ECOG response code definitions
# ------------------------------------------------------------

print("\nC. ECOG RESPONSE CODE DEFINITIONS")

ecog_dictionary = relevant_dic[
    relevant_dic["FLDNAME"].str.upper().str.startswith("ECOGPT")
].copy()

display(
    ecog_dictionary[
        ["PHASE", "FLDNAME", "TEXT", "CODE", "MAPPING_NOTES"]
    ].sort_values("FLDNAME")
)

# ------------------------------------------------------------
# D. Inspect actual response values in the 12 ECOG items
# ------------------------------------------------------------

print("\nD. ACTUAL ECOG RESPONSE VALUES")

ecog_items = [f"ECOGPT{i}" for i in range(1, 13)]

for col in ecog_items:
    print(f"\n{col}")
    display(
        ecog[col]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("Records")
    )

print("\nSTEP 5 COMPLETED")

# ============================================================



A. DUPLICATED DXSUM RECORDS


,PHASE,PTID,RID,VISCODE,VISCODE2,EXAMDATE,DIAGNOSIS,DXNORM,DXNODEP,DXMCI,...,DXODES,DXCONFID,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp
15763,ADNI4,129_S_6452,6452,4_m12,m90,2025-11-19,1.0,NaN,NaN,NaN,...,NaN,NaN,129426,129,2025-11-19,2025-11-19,v1,e,0.0,2025-11-20 00:59:14
16015,ADNI4,129_S_6452,6452,4_m24,m90,2026-04-01,1.0,NaN,NaN,NaN,...,NaN,NaN,141711,129,2026-04-01,2026-04-01,v1,e,0.0,2026-04-02 01:03:26



Values for key diagnosis and date fields:


,PTID,RID,VISCODE,VISCODE2,EXAMDATE,DIAGNOSIS,PHASE
15763,129_S_6452,6452,4_m12,m90,2025-11-19,1.0,ADNI4
16015,129_S_6452,6452,4_m24,m90,2026-04-01,1.0,ADNI4



B. DUPLICATED ECOG PARTICIPANT-VISIT KEYS
Number of duplicated ECOG keys: 0


,PTID,VISCODE2,ECOG_records


... (23 outputs left out)

,Records
ECOGPT11,
1.0,698
2.0,606
3.0,211
4.0,114
9.0,13
NaN,38



ECOGPT12


,Records
ECOGPT12,
1.0,685
2.0,580
3.0,227
4.0,132
9.0,18
NaN,38



STEP 5 COMPLETED


## Visit identifiers and code definitions

Compare visit-code linkage and inspect dictionary definitions, including missing-value codes.


In [44]:
# STEP 6: VERIFY VISIT IDENTIFIERS AND ECOG CODE DEFINITIONS
# ============================================================

# ------------------------------------------------------------
# A. Compare exact participant-visit matching using both codes
# ------------------------------------------------------------

print("A. VISIT MATCHING COMPARISON")

ecog_viscode_pairs = (
    ecog.dropna(subset=["PTID", "VISCODE"])
    [["PTID", "VISCODE"]]
    .drop_duplicates()
)

dx_viscode_pairs = (
    dx4.dropna(subset=["PTID", "VISCODE"])
    [["PTID", "VISCODE"]]
    .drop_duplicates()
)

match_viscode = ecog_viscode_pairs.merge(
    dx_viscode_pairs,
    on=["PTID", "VISCODE"],
    how="inner"
)

ecog_viscode2_pairs = (
    ecog.dropna(subset=["PTID", "VISCODE2"])
    [["PTID", "VISCODE2"]]
    .drop_duplicates()
)

dx_viscode2_pairs = (
    dx4.dropna(subset=["PTID", "VISCODE2"])
    [["PTID", "VISCODE2"]]
    .drop_duplicates()
)

match_viscode2 = ecog_viscode2_pairs.merge(
    dx_viscode2_pairs,
    on=["PTID", "VISCODE2"],
    how="inner"
)

print("Exact PTID + VISCODE matches:", len(match_viscode))
print("Exact PTID + VISCODE2 matches:", len(match_viscode2))

# ------------------------------------------------------------
# B. Inspect the problematic participant in both datasets
# ------------------------------------------------------------

print("\nB. VISIT RECORDS FOR PARTICIPANT 129_S_6452")

display(
    ecog[ecog["PTID"] == "129_S_6452"][
        ["PTID", "VISCODE", "VISCODE2", "COLDATE"]
    ].sort_values("COLDATE")
)

display(
    dx4[dx4["PTID"] == "129_S_6452"][
        ["PTID", "VISCODE", "VISCODE2", "EXAMDATE", "DIAGNOSIS"]
    ].sort_values("EXAMDATE")
)

# ------------------------------------------------------------
# C. Compare visit-code combinations
# ------------------------------------------------------------

print("\nC. ECOG VISCODE AND VISCODE2 COMBINATIONS")

display(
    ecog.groupby(["VISCODE", "VISCODE2"], dropna=False)
    .size()
    .reset_index(name="Records")
    .sort_values(["VISCODE", "VISCODE2"])
)

# ------------------------------------------------------------
# D. Show complete ECOG response code definitions
# ------------------------------------------------------------

print("\nD. COMPLETE ECOG RESPONSE CODE DEFINITIONS")

pd.set_option("display.max_colwidth", None)

display(
    ecog_dictionary[
        ["FLDNAME", "TEXT", "CODE", "MAPPING_NOTES"]
    ].sort_values("FLDNAME").drop_duplicates()
)

# ------------------------------------------------------------
# E. Check whether the dictionary contains a specific code 9
# ------------------------------------------------------------

print("\nE. SEARCH DICTIONARY FOR CODE 9 DEFINITIONS")

code9_dic = dic[
    dic["CODE"].astype(str).str.contains(
        r"(^|[;, ])9\s*=", regex=True, na=False
    )
]

display(
    code9_dic[
        ["PHASE", "CRFNAME", "TBLNAME", "FLDNAME", "CODE", "MAPPING_NOTES"]
    ].drop_duplicates()
)

print("\nSTEP 6 COMPLETED")

# ============================================================


A. VISIT MATCHING COMPARISON
Exact PTID + VISCODE matches: 1622
Exact PTID + VISCODE2 matches: 1620

B. VISIT RECORDS FOR PARTICIPANT 129_S_6452


,PTID,VISCODE,VISCODE2,COLDATE


,PTID,VISCODE,VISCODE2,EXAMDATE,DIAGNOSIS
14316,129_S_6452,4_init,m66,2024-11-19,1.0
15763,129_S_6452,4_m12,m90,2025-11-19,1.0
16015,129_S_6452,4_m24,m90,2026-04-01,1.0



C. ECOG VISCODE AND VISCODE2 COMBINATIONS


,VISCODE,VISCODE2,Records
0,4_bl,bl,535
1,4_init,m108,1
2,4_init,m12,3
3,4_init,m126,3
4,4_init,m132,15
...,...,...,...
95,4_m36,m48,1
96,4_m36,m54,2
97,4_m36,m60,2
98,4_m36,m84,1



D. COMPLETE ECOG RESPONSE CODE DEFINITIONS


,FLDNAME,TEXT,CODE,MAPPING_NOTES
28326,ECOGPT1,1. Remembering where you have placed personal items or objects:,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28327,ECOGPT10,10. Managing bill payments,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28328,ECOGPT11,11. Doing two things at once,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28329,ECOGPT12,12. Working on a particular task while talking to someone at the same time,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28330,ECOGPT2,2. Remembering the current date or day of the week,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28331,ECOGPT3,3. Communicating or expressing ideas in a conversation,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28332,ECOGPT4,4. Understanding spoken directions or instructions,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28333,ECOGPT5,5. Difficulty calculating distances when driving or driving too close to another vehicle,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28334,ECOGPT6,6. Finding your way around a familiar home,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN
28335,ECOGPT7,7. Anticipating weather changes and planning accordingly,1=1- Better or no change; 2=2- Questionable or occasionally worse; 3=3- Consistently a little worse; 4=4- Consistently much worse; 9=9- I don’t know or not applicable,NaN



E. SEARCH DICTIONARY FOR CODE 9 DEFINITIONS


C:\Users\amnar\AppData\Local\Temp\ipykernel_92088\3461314799.py:101: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  dic["CODE"].astype(str).str.contains(


,PHASE,CRFNAME,TBLNAME,FLDNAME,CODE,MAPPING_NOTES
5,ADNI1,ADAS-Cognitive Behavior,ADAS,COT1LIST,0=Butter; 1=Arm; 2=Shore; 3=Letter; 4=Queen; 5=Cabin; 6=Pole; 7=Ticket; 8=Grass; 9=Engine;,Mapped to Q1TR1. Codes shifted down by 1 to match latest phases
6,ADNI1,ADAS-Cognitive Behavior,ADAS,COT2LIST,0=Pole; 1=Letter; 2=Butter; 3=Queen; 4=Arm; 5=Shore; 6=Grass; 7=Cabin; 8=Ticket; 9=Engine;,Mapped to Q1TR2. Codes shifted down by 1 to match latest phases
7,ADNI1,ADAS-Cognitive Behavior,ADAS,COT3LIST,0=Shore; 1=Letter; 2=Arm; 3=Cabin; 4=Pole; 5=Ticket; 6=Engine; 7=Grass; 8=Butter; 9=Queen;,Mapped to Q1TR3. Codes shifted down by 1 to match latest phases
16,ADNI1,ADAS-Cognitive Behavior,ADAS,COT4LIST,0=Butter; 1=Arm; 2=Shore; 3=Letter; 4=Queen; 5=Cabin; 6=Pole; 7=Ticket; 8=Grass; 9=Engine;,Mapped to Q4TASK. Codes shifted down by 1 to match latest phases
19,ADNI1,ADAS-Cognitive Behavior,ADAS,CONAME,1=Flower; 2=Bed; 3=Whistle; 4=Pencil; 5=Rattle; 6=Mask; 7=Scissors; 8=Comb; 9=Wallet; 10=Harmonica; 11=Stethoscope; 12=Tongs; 13=Thumb; 14=Middle; 15=Ring; 16=Index; 17=Pinky; 18=None completed correctly,"Mapped to Q5NAME1, Q5NAME2, Q5NAME3, Q5NAME4, Q5NAME5, Q5NAME6, Q5NAME7, Q5NAME8, Q5NAME9, Q5NAME10, Q5NAME11, Q5NAME12 and Q5FINGER"
...,...,...,...,...,...,...
78882,ADNIGO,Protocol Deviations Log,VIOLOG,VIODES,"1=1. Inclusion Criteria (provide item number below) ;2=2. Exclusion Criteria (provide item number below) ;3=3. Out of Window Baseline Visit ;4=4. Initiation/change of cholinesterase inhibitor or memantine ;5=5. Started Excluded Medication (does not include Cholinesterase Inhibitor or Memantine) ;6=6. Missed Visit ;7=7. Missed Vital Signs ;8=8. Deviation from vitals collection procedures ;9=9. Missed Screening laboratory tests ;10=10. Screening laboratory tests done outside the protocol-required time ;11=11.Deviation from blood sample collection procedures (Biomarkers) ;12=12.Deviation from blood sample collection procedures (ApoE/GWAS/RNA Genoyping) ;13=13.Deviation from blood sample collection procedures (Cell Immortalization) ;14=14. Deviation from CSF Collection Procedures ;15=15. Subject/Study Partner (or legal representative, if applicable) did not sign the initial consent form ;16=16. Subject/Study Partner (or legal representative, if applicable) did not sign updated/renewal consent form (if applicable) ;17=17. Subject data reported prior to signed consent ;18=18. Out-of-window Visit (Does not include out-of-window baseline visit) ;19=19. Out-of-window MRI ;20=20. Out-of-window FDG PET ;21=21. Out-of-window 18F-AV-45 PET ;22=22. Out of mCi dose range FDG ;23=23. Out of mCi dose range AV-45 ;24=24. Missed LP Follow-Up Call ;25=25. Missed AV-45 Follow-Up Call ;26=26. Other",NaN
78899,ADNI2,Protocol Deviations Log,VIOLOG,VIODES,"1=1. Inclusion Criteria (provide item number below) ;2=2. Exclusion Criteria (provide item number below) ;3=3. Out-of-window Baseline Visit ;4=4. Initiation/change of cholinesterase inhibitor or memantine ;5=5. Started Excluded Medication (does not include cholinesterase inhibitor or memantine) ;6=6. Missed Visit ;7=7. Missed Vital Signs ;8=8. Deviation from vitals collection procedures ;9=9. Missed Screening laboratory tests ;10=10. Screening laboratory tests done outside the protocol-required time ;11=11.Deviation from blood sample collection procedures (Biomarkers) ;12=12.Deviation from blood sample collection procedures (ApoE/GWAS/RNA Genotyping) ;13=13.Deviation from blood sample collection procedures (Cell Immortalization) ;14=14. Deviation from CSF collection procedures ;15=15. Subject/Study Partner (or legal representative, if applicable) did not sign the initial consent form ;16=16. Subject/Study Partner (or legal representative, if applicable) did not sign updated/renewal consent form (if applicable) ;17=17. Subject data reported prior to signed consent ;18=18. Out-of-window Visit (Does not include Out-of-window Baseline Visit) ;19=19. Out-of-window MRI ;20=20. Out-of-window FDG PET ;21=21. Out-


STEP 6 COMPLETED


## Validated ECOG-DXSUM dataset

Apply visit matching and ECOG quality rules to build the validated ECOG-DXSUM table.


In [45]:
# STEP 7: CREATE VALIDATED ECOG-DXSUM DATASET
# ============================================================

import numpy as np

# ------------------------------------------------------------
# A. Check uniqueness of PTID + VISCODE
# ------------------------------------------------------------

print("A. CHECKING VISIT-KEY UNIQUENESS")

ecog_key_duplicates = (
    ecog.dropna(subset=["PTID", "VISCODE"])
    .duplicated(subset=["PTID", "VISCODE"], keep=False)
)

dx_key_duplicates = (
    dx4.dropna(subset=["PTID", "VISCODE"])
    .duplicated(subset=["PTID", "VISCODE"], keep=False)
)

print("Duplicate ECOG keys:", ecog_key_duplicates.sum())
print("Duplicate DXSUM keys:", dx_key_duplicates.sum())

if ecog_key_duplicates.any():
    display(
        ecog.loc[ecog_key_duplicates, ["PTID", "VISCODE", "VISCODE2"]]
    )

if dx_key_duplicates.any():
    display(
        dx4.loc[dx_key_duplicates, 
                ["PTID", "VISCODE", "VISCODE2", "EXAMDATE", "DIAGNOSIS"]]
    )

# ------------------------------------------------------------
# B. Prepare DXSUM diagnosis records
# ------------------------------------------------------------

dx_for_merge = dx4[
    ["PTID", "VISCODE", "VISCODE2", "EXAMDATE", "DIAGNOSIS"]
].copy()

ecog_for_merge = ecog.copy()

# Remove rows with missing linkage keys
dx_for_merge = dx_for_merge.dropna(
    subset=["PTID", "VISCODE"]
)

ecog_for_merge = ecog_for_merge.dropna(
    subset=["PTID", "VISCODE"]
)

# ------------------------------------------------------------
# C. Merge only if both keys are unique
# ------------------------------------------------------------

print("\nB. MERGING ECOG WITH DXSUM")

if (
    not ecog_for_merge.duplicated(["PTID", "VISCODE"]).any()
    and not dx_for_merge.duplicated(["PTID", "VISCODE"]).any()
):
    matched_ecog_dx = ecog_for_merge.merge(
        dx_for_merge,
        on=["PTID", "VISCODE"],
        how="inner",
        validate="one_to_one",
        suffixes=("_ECOG", "_DXSUM")
    )
    print("One-to-one merge successful.")
else:
    raise ValueError(
        "Duplicate participant-visit keys detected. "
        "Resolve them before merging."
    )

print("Matched records:", len(matched_ecog_dx))
print("Unique participants:", matched_ecog_dx["PTID"].nunique())

# ------------------------------------------------------------
# D. Diagnosis distribution
# ------------------------------------------------------------

print("\nC. DIAGNOSIS DISTRIBUTION")

display(
    matched_ecog_dx["DIAGNOSIS"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Records")
)

# ------------------------------------------------------------
# E. Clean ECOG responses
# ------------------------------------------------------------

print("\nD. CLEANING ECOG RESPONSES")

ecog_items = [f"ECOGPT{i}" for i in range(1, 13)]

for col in ecog_items:
    matched_ecog_dx[col] = pd.to_numeric(
        matched_ecog_dx[col],
        errors="coerce"
    ).replace(9, np.nan)

# Count valid responses per participant-visit
matched_ecog_dx["ECOG_VALID_ITEMS"] = (
    matched_ecog_dx[ecog_items].notna().sum(axis=1)
)

# Calculate mean only from valid responses
matched_ecog_dx["ECOG12_MEAN"] = (
    matched_ecog_dx[ecog_items].mean(axis=1)
)

matched_ecog_dx["ECOG12_SUM"] = (
    matched_ecog_dx[ecog_items].sum(axis=1, min_count=1)
)

print("ECOG response summary:")
display(
    matched_ecog_dx[
        ["ECOG_VALID_ITEMS", "ECOG12_MEAN", "ECOG12_SUM"]
    ].describe()
)

print("\nNumber of visits with zero valid ECOG responses:",
      (matched_ecog_dx["ECOG_VALID_ITEMS"] == 0).sum())

# ------------------------------------------------------------
# F. Diagnosis eligibility for primary CN vs MCI analysis
# ------------------------------------------------------------

print("\nE. PRIMARY CN-MCI ELIGIBILITY")

matched_ecog_dx["DIAGNOSIS_LABEL"] = (
    matched_ecog_dx["DIAGNOSIS"]
    .map({1: "CN", 2: "MCI", 3: "Dementia"})
)

display(
    matched_ecog_dx["DIAGNOSIS_LABEL"]
    .value_counts(dropna=False)
    .to_frame("Records")
)

# Keep only CN and MCI for the primary classification dataset
ecog_cn_mci = matched_ecog_dx[
    matched_ecog_dx["DIAGNOSIS"].isin([1, 2])
].copy()

print("\nCN/MCI records:", len(ecog_cn_mci))
print("CN/MCI participants:", ecog_cn_mci["PTID"].nunique())

display(
    ecog_cn_mci["DIAGNOSIS_LABEL"]
    .value_counts()
    .to_frame("Records")
)

print("\nSTEP 7 COMPLETED")

# ============================================================


A. CHECKING VISIT-KEY UNIQUENESS
Duplicate ECOG keys: 0
Duplicate DXSUM keys: 0

B. MERGING ECOG WITH DXSUM
One-to-one merge successful.
Matched records: 1622
Unique participants: 1036

C. DIAGNOSIS DISTRIBUTION


,Records
DIAGNOSIS,
1.0,779
2.0,634
3.0,207
NaN,2



D. CLEANING ECOG RESPONSES
ECOG response summary:


,ECOG_VALID_ITEMS,ECOG12_MEAN,ECOG12_SUM
count,1622.000000,1588.000000,1588.000000
mean,11.524044,1.595530,18.696474
std,1.833704,0.563092,6.448951
min,0.000000,1.000000,4.000000
25%,12.000000,1.166667,14.000000
50%,12.000000,1.416667,17.000000
75%,12.000000,1.833333,22.000000
max,12.000000,4.000000,47.000000



Number of visits with zero valid ECOG responses: 34

E. PRIMARY CN-MCI ELIGIBILITY


,Records
DIAGNOSIS_LABEL,
CN,779
MCI,634
Dementia,207
NaN,2



CN/MCI records: 1413
CN/MCI participants: 944


,Records
DIAGNOSIS_LABEL,
CN,779
MCI,634



STEP 7 COMPLETED


## Storyteller participant and visit linkage

Inspect Storyteller identifiers, visit labels, and assessment dates.


In [46]:
# STEP 8: STORYTELLER PARTICIPANT AND VISIT LINKAGE AUDIT
# ============================================================

# ------------------------------------------------------------
# A. Inspect identifier, date and visit-related columns
# ------------------------------------------------------------

print("=" * 75)
print("A. STORYTELLER IDENTIFIER AND DATE COLUMNS")
print("=" * 75)

story_id_date_cols = [
    col for col in story.columns
    if any(term in col.upper() for term in [
        "PTID", "RID", "ONLINEID", "ID",
        "DATE", "TIMEPOINT", "VISIT", "PHASE"
    ])
]

print("Relevant columns:")
print(story_id_date_cols)

display(
    story[story_id_date_cols].head(10)
)

# ------------------------------------------------------------
# B. PTID availability
# ------------------------------------------------------------

print("\nB. STORYTELLER PTID AVAILABILITY")

print("Total records:", len(story))
print("PTID available:", story["PTID"].notna().sum())
print("PTID missing:", story["PTID"].isna().sum())
print("Unique PTIDs:", story["PTID"].nunique())

story_with_ptid = story.dropna(subset=["PTID"]).copy()

# ------------------------------------------------------------
# C. Storyteller participant overlap with ECOG-DXSUM
# ------------------------------------------------------------

print("\nC. PARTICIPANT OVERLAP")

story_ids = set(story_with_ptid["PTID"].unique())
ecog_cn_mci_ids = set(ecog_cn_mci["PTID"].unique())

print("Storyteller participants with PTID:", len(story_ids))
print("ECOG-DXSUM CN/MCI participants:", len(ecog_cn_mci_ids))
print("Shared participants:", len(story_ids & ecog_cn_mci_ids))

print(
    "Storyteller participants without ECOG-DXSUM CN/MCI data:",
    len(story_ids - ecog_cn_mci_ids)
)

# ------------------------------------------------------------
# D. Storyteller timepoint distribution
# ------------------------------------------------------------

print("\nD. STORYTELLER TIMEPOINT DISTRIBUTION")

display(
    story["RMT_Timepoint"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Records")
)

# ------------------------------------------------------------
# E. Check PTID + timepoint duplicates
# ------------------------------------------------------------

print("\nE. PTID + STORYTELLER TIMEPOINT DUPLICATES")

story_key_counts = (
    story_with_ptid
    .groupby(["PTID", "RMT_Timepoint"], dropna=False)
    .size()
    .reset_index(name="Records")
)

story_duplicates = story_key_counts[
    story_key_counts["Records"] > 1
]

print("Duplicate PTID-timepoint combinations:", len(story_duplicates))

display(
    story_duplicates.sort_values(
        "Records", ascending=False
    ).head(30)
)

# ------------------------------------------------------------
# F. Examine relationship between PTID and ADNIOnlineID
# ------------------------------------------------------------

print("\nF. PTID AND ADNIOnlineID RELATIONSHIP")

if "ADNIOnlineID" in story.columns:

    online_id_stats = (
        story.dropna(subset=["PTID", "ADNIOnlineID"])
        .groupby("ADNIOnlineID")["PTID"]
        .nunique()
    )

    ptid_online_stats = (
        story.dropna(subset=["PTID", "ADNIOnlineID"])
        .groupby("PTID")["ADNIOnlineID"]
        .nunique()
    )

    print("Rows with both identifiers:",
          story.dropna(subset=["PTID", "ADNIOnlineID"]).shape[0])

    print("Online IDs associated with multiple PTIDs:",
          (online_id_stats > 1).sum())

    print("PTIDs associated with multiple Online IDs:",
          (ptid_online_stats > 1).sum())

    print("\nExample records with both identifiers:")

    display(
        story.dropna(subset=["PTID", "ADNIOnlineID"])[
            ["PTID", "ADNIOnlineID", "EXAMDATE", "RMT_Timepoint"]
        ].head(20)
    )

# ------------------------------------------------------------
# G. Inspect Storyteller records with known PTIDs
# ------------------------------------------------------------

print("\nG. SAMPLE STORYTELLER RECORDS WITH PTID")

display(
    story_with_ptid[
        story_id_date_cols
    ].head(20)
)

# ------------------------------------------------------------
# H. Examine dates and date coverage
# ------------------------------------------------------------

print("\nH. STORYTELLER DATE COVERAGE")

story["EXAMDATE_PARSED"] = pd.to_datetime(
    story["EXAMDATE"],
    errors="coerce"
)

print("Earliest Storyteller date:",
      story["EXAMDATE_PARSED"].min())

print("Latest Storyteller date:",
      story["EXAMDATE_PARSED"].max())

print("Invalid or missing parsed dates:",
      story["EXAMDATE_PARSED"].isna().sum())

print("\nSTEP 8 COMPLETED")

# ============================================================


A. STORYTELLER IDENTIFIER AND DATE COLUMNS
Relevant columns:
['RMT_PHASE', 'ADNIOnlineID', 'PTID', 'RMT_Timepoint', 'EXAMDATE', 'STORY1IMM_IDEA_DENSITY', 'STORY2IMM_IDEA_DENSITY', 'STORY1DEL_IDEA_DENSITY', 'update_stamp']


,RMT_PHASE,ADNIOnlineID,PTID,RMT_Timepoint,EXAMDATE,STORY1IMM_IDEA_DENSITY,STORY2IMM_IDEA_DENSITY,STORY1DEL_IDEA_DENSITY,update_stamp
0,ADNI4,ADNI4-PYP-9P9,NaN,m00,2023-06-21,0.405405,0.453125,0.365385,2026-02-04 08:17:07
1,ADNI4,ADNI4-PYP-9P9,NaN,m06,2024-01-12,0.360656,0.514286,0.357143,2026-02-04 08:17:07
2,ADNI4,ADNI4-PGP-PNH,NaN,m00,2023-06-28,0.455696,0.485714,0.451613,2026-02-04 08:17:07
3,ADNI4,ADNI4-PGP-PNH,NaN,m06,2024-01-02,0.467391,0.468354,0.500000,2026-02-04 08:17:07
4,ADNI4,ADNI4-PGP-PNH,NaN,m12,2024-07-03,0.473282,0.545455,0.416667,2026-02-04 08:17:07
5,ADNI4,ADNI4-PGP-PNH,NaN,m18,2025-01-07,0.437037,0.509615,0.440299,2026-02-04 08:17:07
6,ADNI4,ADNI4-KQ6-64P,NaN,m00,2023-07-05,0.361702,0.488889,0.390805,2026-02-04 08:17:07
7,ADNI4,ADNI4-KQ6-64P,NaN,m06,2024-01-10,0.461538,0.526316,0.511111,2026-02-04 08:17:07
8,ADNI4,ADNI4-KQ6-64P,NaN,m12,2024-07-08,0.413333,0.556962,0.426471,2026-02-04 08:17:07
9,ADNI4,ADNI4-KQ6-64P,NaN,m18,2025-01-03,0.428571,0.520000,0.430556,2026-02-04 08:17:07



B. STORYTELLER PTID AVAILABILITY
Total records: 15878
PTID available: 1672
PTID missing: 14206
Unique PTIDs: 539

C. PARTICIPANT OVERLAP
Storyteller participants with PTID: 539
ECOG-DXSUM CN/MCI participants: 944
Shared participants: 489
Storyteller participants without ECOG-DXSUM CN/MCI data: 50

D. STORYTELLER TIMEPOINT DISTRIBUTION


,Records
RMT_Timepoint,
m00,8187
m06,4467
m12,2356
m18,679
m24,156
m30,33



E. PTID + STORYTELLER TIMEPOINT DUPLICATES
Duplicate PTID-timepoint combinations: 0


,PTID,RMT_Timepoint,Records



F. PTID AND ADNIOnlineID RELATIONSHIP
Rows with both identifiers: 1672
Online IDs associated with multiple PTIDs: 0
PTIDs associated with multiple Online IDs: 0

Example records with both identifiers:


,PTID,ADNIOnlineID,EXAMDATE,RMT_Timepoint
15,941_S_6254,ADNI4-CYD-KMX,2025-04-09,m18
16,941_S_6454,ADNI4-KX9-WDF,2023-08-30,m00
17,941_S_6454,ADNI4-KX9-WDF,2024-03-11,m06
18,941_S_6454,ADNI4-KX9-WDF,2024-09-04,m12
19,941_S_6454,ADNI4-KX9-WDF,2025-03-02,m18
23,022_S_10043,ADNI4-9HL-QKQ,2023-08-17,m00
24,022_S_10043,ADNI4-9HL-QKQ,2024-08-18,m12
25,022_S_10043,ADNI4-9HL-QKQ,2025-02-26,m18
26,033_S_10005,ADNI4-K3G-3G7,2023-10-30,m00
27,033_S_10005,ADNI4-K3G-3G7,2024-05-05,m06



G. SAMPLE STORYTELLER RECORDS WITH PTID


,RMT_PHASE,ADNIOnlineID,PTID,RMT_Timepoint,EXAMDATE,STORY1IMM_IDEA_DENSITY,STORY2IMM_IDEA_DENSITY,STORY1DEL_IDEA_DENSITY,update_stamp
15,ADNI4,ADNI4-CYD-KMX,941_S_6254,m18,2025-04-09,0.484211,0.452830,0.416667,2026-02-04 08:17:07
16,ADNI4,ADNI4-KX9-WDF,941_S_6454,m00,2023-08-30,0.451613,0.487179,0.468354,2026-02-04 08:17:07
17,ADNI4,ADNI4-KX9-WDF,941_S_6454,m06,2024-03-11,0.456311,0.509434,0.404255,2026-02-04 08:17:07
18,ADNI4,ADNI4-KX9-WDF,941_S_6454,m12,2024-09-04,0.500000,0.479167,0.488722,2026-02-04 08:17:07
19,ADNI4,ADNI4-KX9-WDF,941_S_6454,m18,2025-03-02,0.457364,0.465347,0.444444,2026-02-04 08:17:07
23,ADNI4,ADNI4-9HL-QKQ,022_S_10043,m00,2023-08-17,0.434783,0.470588,0.464286,2026-02-04 08:17:07
24,ADNI4,ADNI4-9HL-QKQ,022_S_10043,m12,2024-08-18,0.444444,0.300000,0.333333,2026-02-04 08:17:07
25,ADNI4,ADNI4-9HL-QKQ,022_S_10043,m18,2025-02-26,0.352941,0.404762,0.360000,2026-02-04 08:17:07
26,ADNI4,ADNI4-K3G-3G7,033_S_10005,m00,2023-10-30,0.476190,0.532710,0.479167,2026-02-04 08:17:07
27,ADNI4,ADNI4-K3G-3G7,033_S_10005,m06,2024-05-05,0.478261,0.480769,0.458824,2026-02-04 08:17:07



H. STORYTELLER DATE COVERAGE
Earliest Storyteller date: 2023-06-21 00:00:00
Latest Storyteller date: 2026-08-11 00:00:00
Invalid or missing parsed dates: 0

STEP 8 COMPLETED


## Storyteller-DXSUM date alignment

Compare Storyteller assessments with diagnosis records available on or before each assessment.


In [47]:
# STEP 9: STORYTELLER-DXSUM DATE AND VISIT ALIGNMENT
# ============================================================

# A. Prepare Storyteller records with known PTIDs
story_linked = story.dropna(subset=["PTID"]).copy()

story_linked["EXAMDATE"] = pd.to_datetime(
    story_linked["EXAMDATE"], errors="coerce"
)

# B. Prepare ADNI4 DXSUM records
dx_date = dx4[
    ["PTID", "VISCODE", "VISCODE2", "EXAMDATE", "DIAGNOSIS"]
].copy()

dx_date["EXAMDATE"] = pd.to_datetime(
    dx_date["EXAMDATE"], errors="coerce"
)

dx_date = dx_date.dropna(
    subset=["PTID", "EXAMDATE"]
)

# C. Match each Storyteller record to all DXSUM visits
candidates = story_linked.merge(
    dx_date,
    on="PTID",
    how="inner",
    suffixes=("_STORY", "_DX")
)

# Positive = diagnosis visit occurred before Storyteller assessment
candidates["DAYS_BEFORE_STORY"] = (
    candidates["EXAMDATE_STORY"] -
    candidates["EXAMDATE_DX"]
).dt.days

print("A. CANDIDATE MATCH COUNTS")
print("Storyteller records with PTID:", len(story_linked))
print("Storyteller-DXSUM candidate pairs:", len(candidates))

# D. Keep only diagnoses on or before the Storyteller date
prior_candidates = candidates[
    candidates["DAYS_BEFORE_STORY"] >= 0
].copy()

# Latest available diagnosis before or on assessment date
latest_prior = (
    prior_candidates
    .sort_values(
        ["ADNIOnlineID", "RMT_Timepoint", "EXAMDATE_DX"]
    )
    .drop_duplicates(
        subset=["ADNIOnlineID", "RMT_Timepoint"],
        keep="last"
    )
)

print("\nB. TEMPORAL DIAGNOSIS MATCHING")
print("Storyteller records with a prior diagnosis:",
      len(latest_prior))
print("Storyteller records without a prior diagnosis:",
      len(story_linked) - len(latest_prior))

# E. Examine time between diagnosis and Storyteller assessment
print("\nC. DAYS BETWEEN DIAGNOSIS AND STORYTELLER")

display(
    latest_prior["DAYS_BEFORE_STORY"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    )
    .to_frame("Days")
)

# F. Categorize date gaps for inspection
latest_prior["DATE_GAP_CATEGORY"] = pd.cut(
    latest_prior["DAYS_BEFORE_STORY"],
    bins=[-1, 30, 90, 180, 365, 730, float("inf")],
    labels=[
        "0-30 days",
        "31-90 days",
        "91-180 days",
        "181-365 days",
        "366-730 days",
        "Over 730 days"
    ]
)

print("\nD. DATE GAP DISTRIBUTION")

display(
    latest_prior["DATE_GAP_CATEGORY"]
    .value_counts()
    .sort_index()
    .to_frame("Records")
)

# G. Diagnosis distribution after temporal matching
print("\nE. DIAGNOSIS DISTRIBUTION")

display(
    latest_prior["DIAGNOSIS"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Records")
)

# H. Compare Storyteller timepoints with DXSUM visit codes
print("\nF. STORYTELLER TIMEPOINT vs DXSUM VISITCODE")

display(
    pd.crosstab(
        latest_prior["RMT_Timepoint"],
        latest_prior["VISCODE"],
        margins=True
    )
)

# I. Inspect examples with date gaps
print("\nG. EXAMPLE MATCHED RECORDS")

display(
    latest_prior[
        [
            "PTID",
            "ADNIOnlineID",
            "RMT_Timepoint",
            "EXAMDATE_STORY",
            "VISCODE",
            "VISCODE2",
            "EXAMDATE_DX",
            "DAYS_BEFORE_STORY",
            "DIAGNOSIS"
        ]
    ]
    .sort_values("DAYS_BEFORE_STORY")
    .head(25)
)

print("\nSTEP 9 COMPLETED")

# ============================================================


A. CANDIDATE MATCH COUNTS
Storyteller records with PTID: 1672
Storyteller-DXSUM candidate pairs: 4265

B. TEMPORAL DIAGNOSIS MATCHING
Storyteller records with a prior diagnosis: 1459
Storyteller records without a prior diagnosis: 213

C. DAYS BETWEEN DIAGNOSIS AND STORYTELLER


,Days
count,1459.000000
mean,168.494859
std,146.224480
min,0.000000
10%,11.000000
25%,34.000000
50%,174.000000
75%,233.000000
90%,379.000000
95%,415.000000



D. DATE GAP DISTRIBUTION


,Records
DATE_GAP_CATEGORY,
0-30 days,336
31-90 days,258
91-180 days,154
181-365 days,513
366-730 days,194
Over 730 days,4



E. DIAGNOSIS DISTRIBUTION


,Records
DIAGNOSIS,
1.0,1098
2.0,361



F. STORYTELLER TIMEPOINT vs DXSUM VISITCODE


VISCODE,4_bl,4_init,4_m12,4_m24,4_sc,All
RMT_Timepoint,,,,,,
m00,138,202,5,0,15,360
m06,145,178,17,0,25,365
m12,103,67,132,2,18,322
m18,59,24,149,0,15,247
m24,17,6,44,59,7,133
m30,0,0,5,27,0,32
All,462,477,352,88,80,1459



G. EXAMPLE MATCHED RECORDS


,PTID,ADNIOnlineID,RMT_Timepoint,EXAMDATE_STORY,VISCODE,VISCODE2,EXAMDATE_DX,DAYS_BEFORE_STORY,DIAGNOSIS
500,137_S_6919,ADNI4-GQR-YQD,m12,2025-02-25,4_m12,m42,2025-02-25,0,2.0
1848,019_S_10106,ADNI4-YKN-WWY,m00,2025-06-03,4_m12,m12,2025-06-03,0,1.0
3590,067_S_10375,ADNI4-3YY-7CG,m06,2024-11-25,4_sc,sc,2024-11-25,0,1.0
2080,305_S_6871,ADNI4-K33-9W4,m12,2024-09-04,4_init,m48,2024-09-04,0,2.0
2227,941_S_6471,ADNI4-NRK-LHQ,m24,2025-10-27,4_m24,m90,2025-10-27,0,1.0
4099,130_S_10490,ADNI4-JKM-DPF,m12,2026-04-23,4_m12,m12,2026-04-23,0,1.0
2774,037_S_10050,ADNI4-64N-7X9,m12,2025-12-16,4_m12,m18,2025-12-16,0,1.0
3177,100_S_4556,ADNI4-J9Q-XLY,m24,2026-03-09,4_m24,m162,2026-03-09,0,1.0
4107,027_S_10520,ADNI4-PGG-XHX,m12,2026-03-24,4_m12,m12,2026-03-23,1,1.0
1291,128_S_2123,ADNI4-PCR-RPQ,m00,2024-08-29,4_init,m168,2024-08-28,1,2.0



STEP 9 COMPLETED


## Storyteller-ECOG date alignment

Audit prior and nearest ECOG dates relative to Storyteller assessments.


In [48]:
# STEP 10: STORYTELLER-ECOG DATE ALIGNMENT
# ============================================================

# A. Prepare Storyteller records
story_date = story.dropna(subset=["PTID"]).copy()

story_date["EXAMDATE"] = pd.to_datetime(
    story_date["EXAMDATE"], errors="coerce"
)

story_date = story_date.dropna(subset=["EXAMDATE"])

# B. Prepare ECOG records
ecog_date = ecog.copy()

ecog_date["COLDATE"] = pd.to_datetime(
    ecog_date["COLDATE"], errors="coerce"
)

ecog_date = ecog_date.dropna(
    subset=["PTID", "COLDATE"]
)

# C. Match records by participant
story_ecog_candidates = story_date.merge(
    ecog_date[
        ["PTID", "VISCODE", "VISCODE2", "COLDATE"]
    ],
    on="PTID",
    how="inner",
    suffixes=("_STORY", "_ECOG")
)

# D. Calculate date differences
story_ecog_candidates["DAYS_STORY_MINUS_ECOG"] = (
    story_ecog_candidates["EXAMDATE"] -
    story_ecog_candidates["COLDATE"]
).dt.days

story_ecog_candidates["ABS_DATE_GAP"] = (
    story_ecog_candidates["DAYS_STORY_MINUS_ECOG"].abs()
)

print("A. MATCHING COUNTS")
print("Storyteller records:", len(story_date))
print("ECOG records with collection dates:", len(ecog_date))
print("Same-participant candidate pairs:",
      len(story_ecog_candidates))

# E. Find nearest ECOG date for each Storyteller assessment
nearest_ecog = (
    story_ecog_candidates
    .sort_values("ABS_DATE_GAP")
    .drop_duplicates(
        subset=["ADNIOnlineID", "RMT_Timepoint"],
        keep="first"
    )
    .copy()
)

print("\nB. NEAREST ECOG DATE GAP")

display(
    nearest_ecog["ABS_DATE_GAP"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    )
    .to_frame("Absolute days")
)

# F. Count nearest matches within different date windows
windows = [0, 7, 14, 30, 60, 90, 180]

window_counts = pd.DataFrame({
    "Maximum date difference (days)": windows,
    "Storyteller records": [
        (nearest_ecog["ABS_DATE_GAP"] <= w).sum()
        for w in windows
    ]
})

print("\nC. NEAREST ECOG MATCHES WITHIN DATE WINDOWS")
display(window_counts)

# G. Direction of date differences
print("\nD. WAS ECOG COLLECTED BEFORE OR AFTER STORYTELLER?")

nearest_ecog["DATE_DIRECTION"] = np.select(
    [
        nearest_ecog["DAYS_STORY_MINUS_ECOG"] > 0,
        nearest_ecog["DAYS_STORY_MINUS_ECOG"] < 0,
        nearest_ecog["DAYS_STORY_MINUS_ECOG"] == 0
    ],
    [
        "ECOG before Storyteller",
        "ECOG after Storyteller",
        "Same day"
    ],
    default="Unknown"
)

display(
    nearest_ecog["DATE_DIRECTION"]
    .value_counts(dropna=False)
    .to_frame("Records")
)

# H. Inspect exact-date matches
print("\nE. EXACT SAME-DAY MATCHES")

exact_matches = nearest_ecog[
    nearest_ecog["ABS_DATE_GAP"] == 0
]

print("Exact same-day records:", len(exact_matches))

display(
    exact_matches[
        [
            "PTID",
            "ADNIOnlineID",
            "RMT_Timepoint",
            "EXAMDATE",
            "VISCODE",
            "VISCODE2",
            "COLDATE",
            "DAYS_STORY_MINUS_ECOG"
        ]
    ].head(25)
)

# I. Inspect nearest matches with gaps up to 30 days
print("\nF. NEAREST MATCHES WITHIN 30 DAYS")

display(
    nearest_ecog[
        nearest_ecog["ABS_DATE_GAP"] <= 30
    ][
        [
            "PTID",
            "ADNIOnlineID",
            "RMT_Timepoint",
            "EXAMDATE",
            "VISCODE",
            "VISCODE2",
            "COLDATE",
            "DAYS_STORY_MINUS_ECOG"
        ]
    ].head(30)
)

# J. Check diagnosis codes in the temporal candidate dataset
print("\nG. DIAGNOSIS CODE CHECK")

print("All prior diagnosis candidates:")
display(
    prior_candidates["DIAGNOSIS"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Records")
)

print("Selected latest-prior diagnoses:")
display(
    latest_prior["DIAGNOSIS"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Records")
)

print("\nSTEP 10 COMPLETED")

# ============================================================


A. MATCHING COUNTS
Storyteller records: 1672
ECOG records with collection dates: 1676
Same-participant candidate pairs: 2506



B. NEAREST ECOG DATE GAP


,Absolute days
count,1545.000000
mean,191.849838
std,152.013590
min,0.000000
10%,22.000000
25%,59.000000
50%,163.000000
75%,286.000000
90%,400.600000
95%,467.400000



C. NEAREST ECOG MATCHES WITHIN DATE WINDOWS


,Maximum date difference (days),Storyteller records
0,0,2
1,7,41
2,14,91
3,30,202
4,60,392
5,90,491
6,180,830



D. WAS ECOG COLLECTED BEFORE OR AFTER STORYTELLER?


... (6 outputs left out)

,Records
DIAGNOSIS,
1.0,1929
2.0,657


Selected latest-prior diagnoses:


,Records
DIAGNOSIS,
1.0,1098
2.0,361



STEP 10 COMPLETED


## Strict three-way visit alignment

Build and inspect Storyteller, ECOG, and DXSUM combinations under the temporal rules in the source.


In [49]:
# STEP 11: STRICT STORYTELLER-ECOG-DXSUM VISIT ALIGNMENT
# ============================================================

# A. Prepare Storyteller records
story_step11 = story.dropna(subset=["PTID"]).copy()

story_step11["EXAMDATE"] = pd.to_datetime(
    story_step11["EXAMDATE"], errors="coerce"
)

# B. Prepare verified ECOG-DXSUM visit pairs
paired_clinical = matched_ecog_dx.copy()

paired_clinical["COLDATE"] = pd.to_datetime(
    paired_clinical["COLDATE"], errors="coerce"
)

paired_clinical["EXAMDATE"] = pd.to_datetime(
    paired_clinical["EXAMDATE"], errors="coerce"
)

paired_clinical = paired_clinical.dropna(
    subset=["PTID", "COLDATE", "EXAMDATE", "DIAGNOSIS"]
)

# C. Match Storyteller with verified clinical visits by participant
three_way_candidates = story_step11.merge(
    paired_clinical,
    on="PTID",
    how="inner",
    suffixes=("_STORY", "_CLINICAL")
)

# D. Calculate date gaps
three_way_candidates["STORY_MINUS_ECOG"] = (
    three_way_candidates["EXAMDATE_STORY"] -
    three_way_candidates["COLDATE"]
).dt.days

three_way_candidates["STORY_MINUS_DX"] = (
    three_way_candidates["EXAMDATE_STORY"] -
    three_way_candidates["EXAMDATE_CLINICAL"]
).dt.days

three_way_candidates["ABS_ECOG_GAP"] = (
    three_way_candidates["STORY_MINUS_ECOG"].abs()
)

three_way_candidates["ABS_DX_GAP"] = (
    three_way_candidates["STORY_MINUS_DX"].abs()
)

print("A. THREE-WAY CANDIDATES")
print("Candidate Storyteller-clinical visit pairs:",
      len(three_way_candidates))

# E. Require both clinical records to be before/on Storyteller
# and both to be within 30 days
eligible_30 = three_way_candidates[
    (three_way_candidates["STORY_MINUS_ECOG"].between(0, 30)) &
    (three_way_candidates["STORY_MINUS_DX"].between(0, 30))
].copy()

print("\nB. STRICT 30-DAY MATCHES")
print("Eligible three-way visit pairs:", len(eligible_30))
print("Unique Storyteller assessments:",
      eligible_30["ADNIOnlineID"].nunique())

# F. Inspect date gap distributions
print("\nC. DATE GAP SUMMARY")

display(
    eligible_30[
        ["STORY_MINUS_ECOG", "STORY_MINUS_DX"]
    ].describe()
)

# G. Check whether more than one clinical visit matches a Storyteller assessment
eligible_30["STORY_KEY"] = (
    eligible_30["ADNIOnlineID"].astype(str) + "_" +
    eligible_30["RMT_Timepoint"].astype(str)
)

matches_per_story = eligible_30.groupby("STORY_KEY").size()

print("\nD. MULTIPLE CLINICAL VISITS WITHIN 30 DAYS")
print("Storyteller assessments with one eligible visit:",
      (matches_per_story == 1).sum())
print("Storyteller assessments with multiple eligible visits:",
      (matches_per_story > 1).sum())

# H. Select the closest eligible clinical visit per Storyteller assessment
# The smallest maximum gap to ECOG and DXSUM is preferred.
eligible_30["MAX_DATE_GAP"] = eligible_30[
    ["STORY_MINUS_ECOG", "STORY_MINUS_DX"]
].max(axis=1)

eligible_30["TOTAL_DATE_GAP"] = (
    eligible_30["STORY_MINUS_ECOG"] +
    eligible_30["STORY_MINUS_DX"]
)

best_three_way = (
    eligible_30
    .sort_values(
        ["MAX_DATE_GAP", "TOTAL_DATE_GAP"]
    )
    .drop_duplicates("STORY_KEY", keep="first")
    .copy()
)

print("\nE. BEST MATCHED THREE-WAY RECORDS")
print("Selected records:", len(best_three_way))
print("Unique participants:", best_three_way["PTID"].nunique())

# I. Diagnosis distribution
print("\nF. DIAGNOSIS DISTRIBUTION")

display(
    best_three_way["DIAGNOSIS"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Records")
)

# J. Inspect selected records
print("\nG. SAMPLE SELECTED MATCHES")

display(
    best_three_way[
        [
            "PTID",
            "ADNIOnlineID",
            "RMT_Timepoint",
            "EXAMDATE_STORY",
            "VISCODE",
            "COLDATE",
            "EXAMDATE_CLINICAL",
            "STORY_MINUS_ECOG",
            "STORY_MINUS_DX",
            "DIAGNOSIS"
        ]
    ].head(30)
)

# K. Investigate why dementia was absent from prior diagnosis matches
print("\nH. DXSUM DIAGNOSIS COUNTS")

display(
    dx4["DIAGNOSIS"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("Records")
)

print("\nDementia records in verified ECOG-DXSUM pairs:")
print(
    (matched_ecog_dx["DIAGNOSIS"] == 3).sum()
)

print("\nDementia records in Storyteller prior diagnosis candidates:")
print(
    (prior_candidates["DIAGNOSIS"] == 3).sum()
)

print("\nSTEP 11 COMPLETED")

# ============================================================


A. THREE-WAY CANDIDATES
Candidate Storyteller-clinical visit pairs: 2401

B. STRICT 30-DAY MATCHES
Eligible three-way visit pairs: 151
Unique Storyteller assessments: 141

C. DATE GAP SUMMARY


,STORY_MINUS_ECOG,STORY_MINUS_DX
count,151.000000,151.000000
mean,16.072848,12.437086
std,7.418984,7.512280
min,1.000000,0.000000
25%,9.500000,6.000000
50%,16.000000,11.000000
75%,22.000000,18.000000
max,30.000000,30.000000



D. MULTIPLE CLINICAL VISITS WITHIN 30 DAYS
Storyteller assessments with one eligible visit: 151
Storyteller assessments with multiple eligible visits: 0

E. BEST MATCHED THREE-WAY RECORDS
Selected records: 151
Unique participants: 141

F. DIAGNOSIS DISTRIBUTION


,Records
DIAGNOSIS,
1.0,105
2.0,46



G. SAMPLE SELECTED MATCHES


,PTID,ADNIOnlineID,RMT_Timepoint,EXAMDATE_STORY,VISCODE,COLDATE,EXAMDATE_CLINICAL,STORY_MINUS_ECOG,STORY_MINUS_DX,DIAGNOSIS
753,128_S_2123,ADNI4-PCR-RPQ,m00,2024-08-29,4_init,2024-08-28,2024-08-28,1,1,2.0
1111,027_S_2245,ADNI4-MCL-YCW,m12,2025-07-02,4_m12,2025-07-01,2025-07-01,1,1,2.0
962,036_S_10642,ADNI4-36N-XN4,m00,2025-04-25,4_bl,2025-04-22,2025-04-23,3,2,1.0
562,027_S_5083,ADNI4-KJR-YCP,m00,2024-06-06,4_init,2024-06-03,2024-06-03,3,3,1.0
1499,082_S_10840,ADNI4-3FJ-7C7,m00,2025-08-14,4_bl,2025-08-11,2025-08-11,3,3,2.0
959,036_S_10595,ADNI4-Y6H-M9N,m00,2025-04-13,4_bl,2025-04-09,2025-04-10,4,3,1.0
527,005_S_10288,ADNI4-Y34-J9G,m06,2024-11-11,4_bl,2024-11-07,2024-11-07,4,4,1.0
908,114_S_10383,ADNI4-KPQ-LCD,m00,2025-01-18,4_bl,2025-01-14,2025-01-14,4,4,1.0
2067,135_S_6359,ADNI4-QLC-4H3,m24,2026-06-19,4_m24,2026-06-15,2026-06-15,4,4,1.0
620,130_S_6688,ADNI4-LDN-MLY,m00,2024-06-30,4_init,2024-06-25,2024-06-25,5,5,1.0



H. DXSUM DIAGNOSIS COUNTS


,Records
DIAGNOSIS,
1.0,1715
2.0,888
3.0,316
NaN,6



Dementia records in verified ECOG-DXSUM pairs:
207

Dementia records in Storyteller prior diagnosis candidates:
0

STEP 11 COMPLETED


## Final cohort quality and diagnosis audit

Review cohort size, diagnosis balance, ECOG completeness, and date gaps.


In [50]:
# STEP 12: FINAL COHORT QUALITY AND DIAGNOSIS AUDIT
# ============================================================

cohort_audit = best_three_way.copy()

# ------------------------------------------------------------
# A. Check unique assessments and repeated participants
# ------------------------------------------------------------

print("A. COHORT STRUCTURE")

print("Assessment records:", len(cohort_audit))
print("Unique participants:", cohort_audit["PTID"].nunique())

print("Duplicate participant-timepoint keys:",
      cohort_audit.duplicated(
          ["PTID", "RMT_Timepoint"]
      ).sum())

visits_per_participant = cohort_audit.groupby("PTID").size()

print("Participants with one assessment:",
      (visits_per_participant == 1).sum())

print("Participants with multiple assessments:",
      (visits_per_participant > 1).sum())

display(
    visits_per_participant.value_counts()
    .sort_index()
    .to_frame("Participants")
)

# ------------------------------------------------------------
# B. Assessment-level and participant-level diagnosis balance
# ------------------------------------------------------------

print("\nB. DIAGNOSIS BALANCE")

cohort_audit["DIAGNOSIS_LABEL"] = (
    cohort_audit["DIAGNOSIS"]
    .map({1: "CN", 2: "MCI", 3: "Dementia"})
)

display(
    cohort_audit["DIAGNOSIS_LABEL"]
    .value_counts()
    .to_frame("Assessments")
)

participant_labels = (
    cohort_audit.groupby("PTID")["DIAGNOSIS_LABEL"]
    .agg(
        number_of_labels="nunique",
        labels=lambda x: ", ".join(sorted(x.dropna().unique()))
    )
)

print("Participants with more than one diagnosis label:",
      (participant_labels["number_of_labels"] > 1).sum())

display(
    participant_labels["labels"]
    .value_counts()
    .to_frame("Participants")
)

# ------------------------------------------------------------
# C. ECOG item completeness
# ------------------------------------------------------------

print("\nC. ECOG COMPLETENESS")

ecog_items = [f"ECOGPT{i}" for i in range(1, 13)]

# Work on a copy; convert code 9 to missing
ecog_values = cohort_audit[ecog_items].apply(
    pd.to_numeric, errors="coerce"
).replace(9, np.nan)

cohort_audit["ECOG_VALID_ITEMS_CHECK"] = (
    ecog_values.notna().sum(axis=1)
)

print("ECOG visits with zero valid items:",
      (cohort_audit["ECOG_VALID_ITEMS_CHECK"] == 0).sum())

print("ECOG visits with all 12 items:",
      (cohort_audit["ECOG_VALID_ITEMS_CHECK"] == 12).sum())

display(
    cohort_audit["ECOG_VALID_ITEMS_CHECK"]
    .value_counts()
    .sort_index()
    .to_frame("Assessments")
)

print("\nECOG item missingness (%)")

display(
    (ecog_values.isna().mean() * 100)
    .round(2)
    .to_frame("Missing percentage")
)

# ------------------------------------------------------------
# D. Storyteller timepoint distribution
# ------------------------------------------------------------

print("\nD. STORYTELLER TIMEPOINTS")

display(
    pd.crosstab(
        cohort_audit["RMT_Timepoint"],
        cohort_audit["DIAGNOSIS_LABEL"],
        margins=True
    )
)

# ------------------------------------------------------------
# E. Investigate dementia diagnoses for selected participants
# ------------------------------------------------------------

print("\nE. DEMENTIA HISTORY CHECK")

selected_story = cohort_audit[
    ["PTID", "ADNIOnlineID", "RMT_Timepoint", "EXAMDATE_STORY"]
].drop_duplicates().copy()

selected_story["EXAMDATE_STORY"] = pd.to_datetime(
    selected_story["EXAMDATE_STORY"]
)

dx_history = dx4[
    ["PTID", "EXAMDATE", "DIAGNOSIS", "VISCODE"]
].copy()

dx_history["EXAMDATE"] = pd.to_datetime(
    dx_history["EXAMDATE"], errors="coerce"
)

dx_history = dx_history.dropna(
    subset=["PTID", "EXAMDATE", "DIAGNOSIS"]
)

history_candidates = selected_story.merge(
    dx_history,
    on="PTID",
    how="left"
)

history_candidates["DX_DATE_RELATIVE_TO_STORY"] = (
    history_candidates["EXAMDATE"] -
    history_candidates["EXAMDATE_STORY"]
).dt.days

dementia_history = history_candidates[
    history_candidates["DIAGNOSIS"] == 3
].copy()

print("Dementia diagnosis records for selected participants:",
      len(dementia_history))

print("Dementia diagnoses before/on Storyteller date:",
      (dementia_history["DX_DATE_RELATIVE_TO_STORY"] <= 0).sum())

print("Dementia diagnoses after Storyteller date:",
      (dementia_history["DX_DATE_RELATIVE_TO_STORY"] > 0).sum())

print("Unique participants with any dementia diagnosis:",
      dementia_history["PTID"].nunique())

print("\nExample dementia history records:")

display(
    dementia_history[
        [
            "PTID",
            "RMT_Timepoint",
            "EXAMDATE_STORY",
            "VISCODE",
            "EXAMDATE",
            "DX_DATE_RELATIVE_TO_STORY"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# F. Final cohort date gap check
# ------------------------------------------------------------

print("\nF. DATE GAP VALIDATION")

print("ECOG collected after Storyteller:",
      (cohort_audit["STORY_MINUS_ECOG"] < 0).sum())

print("Diagnosis after Storyteller:",
      (cohort_audit["STORY_MINUS_DX"] < 0).sum())

print("ECOG gap over 30 days:",
      (cohort_audit["STORY_MINUS_ECOG"] > 30).sum())

print("Diagnosis gap over 30 days:",
      (cohort_audit["STORY_MINUS_DX"] > 30).sum())

print("\nSTEP 12 COMPLETED")



A. COHORT STRUCTURE
Assessment records: 151
Unique participants: 141
Duplicate participant-timepoint keys: 0
Participants with one assessment: 131
Participants with multiple assessments: 10


,Participants
1,131
2,10



B. DIAGNOSIS BALANCE


,Assessments
DIAGNOSIS_LABEL,
CN,105
MCI,46


Participants with more than one diagnosis label: 0


,Participants
labels,
CN,99
MCI,42


... (5 outputs left out)

DIAGNOSIS_LABEL,CN,MCI,All
RMT_Timepoint,,,
m00,87,35,122
m06,5,1,6
m12,5,9,14
m24,8,1,9
All,105,46,151



E. DEMENTIA HISTORY CHECK
Dementia diagnosis records for selected participants: 0
Dementia diagnoses before/on Storyteller date: 0
Dementia diagnoses after Storyteller date: 0
Unique participants with any dementia diagnosis: 0

Example dementia history records:


,PTID,RMT_Timepoint,EXAMDATE_STORY,VISCODE,EXAMDATE,DX_DATE_RELATIVE_TO_STORY



F. DATE GAP VALIDATION
ECOG collected after Storyteller: 0
Diagnosis after Storyteller: 0
ECOG gap over 30 days: 0
Diagnosis gap over 30 days: 0

STEP 12 COMPLETED


## Metadata and feature inventory

Inspect assessment metadata, Storyteller fields, dictionary coverage, and candidate predictors.


In [51]:
# STEP 13: Inspect ECOG assessment metadata and Storyteller features

dictionary = data["DATADIC"].copy()

# A. Inspect definitions of ECOG assessment metadata fields
ecog_metadata_fields = [
    "VALIDITY", "STAFFASST", "SOURCE",
    "CONCERN", "MEMREF"
]

print("=" * 70)
print("A. ECOG ASSESSMENT METADATA DEFINITIONS")
print("=" * 70)

metadata_definitions = dictionary[
    dictionary["FLDNAME"].astype(str).str.upper().isin(ecog_metadata_fields)
]

display(
    metadata_definitions[
        ["PHASE", "CRFNAME", "TBLNAME", "FLDNAME",
         "TEXT", "CODE", "MAPPING_NOTES"]
    ].drop_duplicates()
)

# B. Inspect actual metadata values in the selected cohort
print("\n" + "=" * 70)
print("B. ECOG METADATA VALUE DISTRIBUTIONS")
print("=" * 70)

metadata_columns = [
    col for col in ecog_metadata_fields
    if col in best_three_way.columns
]

for col in metadata_columns:
    print(f"\n{col}:")
    display(
        best_three_way[col]
        .value_counts(dropna=False)
        .rename("Assessments")
        .to_frame()
    )

# C. Inspect all Storyteller-related columns in the selected cohort
print("\n" + "=" * 70)
print("C. STORYTELLER-RELATED COLUMNS")
print("=" * 70)

story_columns = [
    col for col in best_three_way.columns
    if "STORY" in str(col).upper()
    or str(col).upper().startswith("RMT_")
]

for col in story_columns:
    print(col, "|", best_three_way[col].dtype)

# D. Inspect the original Storyteller feature columns
print("\n" + "=" * 70)
print("D. ORIGINAL STORYTELLER COLUMN INVENTORY")
print("=" * 70)

for i, col in enumerate(story.columns, start=1):
    print(f"{i:02d}. {col}")

# E. Identify possible screening and administrative fields
print("\n" + "=" * 70)
print("E. POTENTIAL SCREENING / ADMINISTRATIVE FIELDS")
print("=" * 70)

possible_nonfeatures = [
    col for col in story.columns
    if any(term in str(col).upper() for term in [
        "PTID", "RID", "ID", "DATE", "TIMEPOINT",
        "PHASE", "SCREENER", "UPDATE", "STATUS"
    ])
]

print(possible_nonfeatures)

# STEP 14: Complete Storyteller feature inventory

print("=" * 70)
print("A. ALL STORYTELLER FEATURES IN THE SELECTED COHORT")
print("=" * 70)

story_feature_columns = [
    col for col in story.columns
    if (
        col.upper().startswith("STORY")
        and col.upper() not in ["SCREENER_SCORE", "SCREENER_RECOMMEND"]
    )
]

for i, col in enumerate(story_feature_columns, start=1):
    print(f"{i:02d}. {col}")

print("\nTotal Storyteller-related columns:", len(story_feature_columns))

print("\n" + "=" * 70)
print("B. DATA TYPES AND MISSINGNESS IN THE SELECTED COHORT")
print("=" * 70)

feature_audit = pd.DataFrame({
    "dtype": best_three_way[story_feature_columns].dtypes.astype(str),
    "missing_count": best_three_way[story_feature_columns].isna().sum(),
    "missing_percent": (
        best_three_way[story_feature_columns].isna().mean() * 100
    ).round(2),
    "unique_values": best_three_way[story_feature_columns].nunique(
        dropna=True
    )
})

display(feature_audit)

print("\n" + "=" * 70)
print("C. STORYTELLER SCREENING FIELDS")
print("=" * 70)

for col in ["SCREENER_SCORE", "SCREENER_RECOMMEND"]:
    if col in story.columns:
        print(f"\n{col}")
        display(story[col].value_counts(dropna=False).to_frame("Count"))

# STEP 15: Verify Storyteller feature definitions and investigate special fields

dictionary = data["DATADIC"].copy()

# A. Retrieve dictionary definitions for the 61 Storyteller columns
story_dictionary = dictionary[
    (dictionary["PHASE"].astype(str).str.upper() == "ADNI4") &
    (dictionary["FLDNAME"].astype(str).isin(story_feature_columns))
].copy()

print("=" * 70)
print("A. ADNI4 STORYTELLER FEATURE DEFINITIONS")
print("=" * 70)

display(
    story_dictionary[
        ["CRFNAME", "TBLNAME", "FLDNAME", "TEXT",
         "TYPE", "CODE", "UNITS", "MAPPING_NOTES"]
    ].drop_duplicates().sort_values("FLDNAME")
)

print("\nDictionary definitions found:", story_dictionary["FLDNAME"].nunique())
print("Features in our inventory:", len(story_feature_columns))

# B. Investigate language and playback fields
print("\n" + "=" * 70)
print("B. LANGUAGE AND PLAYBACK VALUES")
print("=" * 70)

special_columns = [
    col for col in story_feature_columns
    if col.endswith("_LANGUAGE") or col.endswith("_PLAYBACK_COUNT")
]

for col in special_columns:
    print(f"\n{col}")
    display(best_three_way[col].value_counts(dropna=False).to_frame("Count"))

# C. Inspect GMATCH-related features
print("\n" + "=" * 70)
print("C. GMATCH FEATURES")
print("=" * 70)

gmatch_columns = [
    col for col in story_feature_columns
    if "GMATCH" in col.upper()
]

display(best_three_way[gmatch_columns].describe().T)

# D. Check missingness patterns across the three response sections
print("\n" + "=" * 70)
print("D. MISSINGNESS BY STORYTELLER RESPONSE SECTION")
print("=" * 70)

for section in ["STORY1IMM", "STORY2IMM", "STORY1DEL"]:
    cols = [col for col in story_feature_columns if col.startswith(section)]

    print(f"\n{section}: {len(cols)} features")

    display(
        pd.DataFrame({
            "Missing": best_three_way[cols].isna().sum(),
            "Missing_percent": (
                best_three_way[cols].isna().mean() * 100
            ).round(2)
        })
    )

# E. Identify assessments with missing delayed-recall features
print("\n" + "=" * 70)
print("E. ASSESSMENTS WITH MISSING STORY1DEL FEATURES")
print("=" * 70)

delayed_columns = [
    col for col in story_feature_columns
    if col.startswith("STORY1DEL")
]

delayed_missing = best_three_way[
    best_three_way[delayed_columns].isna().any(axis=1)
]

print("Assessments with at least one missing delayed feature:",
      len(delayed_missing))

display(
    delayed_missing[
        ["PTID", "RMT_Timepoint", "EXAMDATE_STORY"] + delayed_columns
    ]
)



A. ECOG ASSESSMENT METADATA DEFINITIONS


,PHASE,CRFNAME,TBLNAME,FLDNAME,TEXT,CODE,MAPPING_NOTES
536,ADNI3,ADAS-Cognitive Behavior,ADAS,SOURCE,Information Source,1=Participant Visit;2=Telephone Call/Video Call,NaN
654,ADNI4,ADAS-Cognitive Behavior,ADAS,SOURCE,Information Source,1=Participant Visit; 2=Telephone Call/Video Call,NaN
6471,TEAM,Adverse Events Log,ADVERSE,SOURCE,Information Source,1=Participant Visit; 2=Telephone Call/Video Call,NaN
6743,ADNI4,American National Adult Reading Test,AMNART,SOURCE,Information Source,1=Participant Visit; 2=Telephone Call/Video Call,NaN
6809,TEAM,American National Adult Reading Test,AMNART,SOURCE,Information Source,1=Participant Visit; 2=Telephone Call/Video Call,NaN
6850,ADNI4,Amyloid Disclosure Visit,AMYDISC,SOURCE,In what setting was the disclosure conducted?,1=Remotely; 2=In-Person,NaN
6888,TEAM,Amyloid Disclosure Visit,AMYDISC,SOURCE,In what setting was the disclosure conducted?,1=Remotely; 2=In-Person,NaN
9688,ADNI3,Cognitive Change Index,CCI,SOURCE,Information Source,1=Participant Visit;2=Telephone Call/Video Call,NaN
28325,ADNI4,Everyday Cognition 12 Item - Participant Self-Report,ECOG12PT,CONCERN,Are you concerned that you have a memory or other thinking problem?,0=No; 1=Yes,NaN
28338,ADNI4,Everyday Cognition 12 Item - Participant Self-Report,ECOG12PT,MEMREF,"Compared to one year ago, do you feel that your memory has declined substantially?",0=No; 1=Yes,NaN



B. ECOG METADATA VALUE DISTRIBUTIONS

VALIDITY:


,Assessments
VALIDITY,
1.0,145
2.0,3
NaN,3



STAFFASST:


,Assessments
STAFFASST,
1.0,143
2.0,5
NaN,3


... (32 outputs left out)


STORY1DEL: 21 features


,Missing,Missing_percent
STORY1DEL_LANGUAGE,1,0.66
STORY1DEL_PLAYBACK_COUNT,1,0.66
STORY1DEL_AUDIO_LENGTH,1,0.66
STORY1DEL_NUM_PRONOUNS,1,0.66
STORY1DEL_NUM_NOUNS,1,0.66
STORY1DEL_NUM_PROPER_NOUNS,1,0.66
STORY1DEL_NUM_VERBS,1,0.66
STORY1DEL_NUM_ADVERBS,1,0.66
STORY1DEL_NUM_ADJECTIVES,1,0.66
STORY1DEL_NUM_WORDS,1,0.66



E. ASSESSMENTS WITH MISSING STORY1DEL FEATURES
Assessments with at least one missing delayed feature: 5


,PTID,RMT_Timepoint,EXAMDATE_STORY,STORY1DEL_LANGUAGE,STORY1DEL_PLAYBACK_COUNT,STORY1DEL_AUDIO_LENGTH,STORY1DEL_NUM_PRONOUNS,STORY1DEL_NUM_NOUNS,STORY1DEL_NUM_PROPER_NOUNS,STORY1DEL_NUM_VERBS,...,STORY1DEL_UNIQUE_WORDS_RATIO,STORY1DEL_MEAN_LETTER_PER_WORD,STORY1DEL_GMATCH,STORY1DEL_GMATCH_IMM_DEL,STORY1DEL_SPEECH_DURATION,STORY1DEL_NONSPEECH_PROPORTION,STORY1DEL_SPEECH_RATE,STORY1DEL_ARTICULATION_RATE,STORY1DEL_NUM_PAUSES,STORY1DEL_MEAN_PAUSE_LENGTH
1499,082_S_10840,m00,2025-08-14,en-US,0.0,4.30,0.0,0.0,0.0,0.0,...,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN
1256,099_S_10804,m00,2025-07-13,en-US,0.0,9.16,0.0,0.0,0.0,0.0,...,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN
345,098_S_10358,m12,2025-02-17,en-US,0.0,63.78,0.0,0.0,0.0,0.0,...,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN
661,135_S_5113,m00,2024-08-04,en-US,1.0,25.00,0.0,0.0,0.0,0.0,...,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN
701,099_S_6396,m00,2024-08-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Response completeness and zero-word audits

Review dictionary coverage, response completeness, zero-word rows, and response-quality flags.


In [52]:
# STEP 16: Investigate Storyteller dictionary coverage and response completeness

dictionary = data["DATADIC"].copy()

# A. Search dictionary descriptions and table names
print("=" * 70)
print("A. STORYTELLER / NOVOIC / RMT DICTIONARY SEARCH")
print("=" * 70)

search_columns = ["CRFNAME", "TBLNAME", "FLDNAME", "TEXT"]
search_terms = r"STORYTELLER|NOVOIC|RMT|RECALL|IDEA DENSITY"

dictionary_text = (
    dictionary[search_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

story_dictionary_candidates = dictionary[
    dictionary_text.str.contains(search_terms, case=False, regex=True)
].copy()

print("Matching dictionary entries:",
      len(story_dictionary_candidates))

display(
    story_dictionary_candidates[
        ["PHASE", "CRFNAME", "TBLNAME", "FLDNAME",
         "TEXT", "TYPE", "CODE", "UNITS"]
    ].drop_duplicates().head(100)
)

# B. Check ECOG validity against valid-item counts
print("\n" + "=" * 70)
print("B. ECOG VALIDITY VS RESPONSE COMPLETENESS")
print("=" * 70)

display(
    pd.crosstab(
        best_three_way["VALIDITY"],
        best_three_way["ECOG_VALID_ITEMS"],
        margins=True,
        dropna=False
    )
)

# C. Examine the five assessments with missing delayed-recall features
print("\n" + "=" * 70)
print("C. INCOMPLETE DELAYED-RECALL ASSESSMENTS")
print("=" * 70)

delayed_columns = [
    col for col in story_feature_columns
    if col.startswith("STORY1DEL")
]

delayed_missing = best_three_way[
    best_three_way[delayed_columns].isna().any(axis=1)
].copy()

inspection_columns = [
    "PTID", "RMT_Timepoint", "EXAMDATE_STORY",
    "DIAGNOSIS", "VALIDITY", "ECOG_VALID_ITEMS"
] + delayed_columns

inspection_columns = [
    col for col in inspection_columns
    if col in delayed_missing.columns
]

display(delayed_missing[inspection_columns].T)

# D. Compare immediate and delayed response availability
print("\n" + "=" * 70)
print("D. RESPONSE AVAILABILITY")
print("=" * 70)

availability = pd.DataFrame(index=best_three_way.index)

for section in ["STORY1IMM", "STORY2IMM", "STORY1DEL"]:
    cols = [
        col for col in story_feature_columns
        if col.startswith(section)
    ]

    availability[section + "_missing_features"] = (
        best_three_way[cols].isna().sum(axis=1)
    )

    availability[section + "_all_missing"] = (
        best_three_way[cols].isna().all(axis=1)
    )

display(availability.sum().to_frame("Count"))

# E. Inspect playback counts alongside response availability
print("\n" + "=" * 70)
print("E. PLAYBACK COUNT VS RESPONSE AVAILABILITY")
print("=" * 70)

playback_columns = [
    col for col in story_feature_columns
    if col.endswith("_PLAYBACK_COUNT")
]

for col in playback_columns:
    section = col.replace("_PLAYBACK_COUNT", "")

    response_col = section + "_all_missing"

    if response_col in availability.columns:
        print(f"\n{col}")
        display(pd.crosstab(
            best_three_way[col],
            availability[response_col],
            margins=True,
            dropna=False
        ))

# STEP 17: Focused Storyteller dictionary and zero-word investigation

dictionary = data["DATADIC"].copy()

print("=" * 70)
print("A. EXACT STORYTELLER / NOVOIC / RMT DICTIONARY SEARCH")
print("=" * 70)

for term in ["STORYTELLER", "NOVOIC", "RMT"]:
    print(f"\nSearch term: {term}")

    mask = (
        dictionary["CRFNAME"].astype(str).str.contains(
            term, case=False, na=False
        ) |
        dictionary["TBLNAME"].astype(str).str.contains(
            term, case=False, na=False
        ) |
        dictionary["FLDNAME"].astype(str).str.contains(
            term, case=False, na=False
        )
    )

    results = dictionary.loc[mask]

    print("Matching rows:", len(results))
    print("Matching tables:",
          results["TBLNAME"].dropna().unique()[:30])

    display(
        results[
            ["PHASE", "CRFNAME", "TBLNAME", "FLDNAME", "TEXT"]
        ].drop_duplicates().head(40)
    )


# B. Inspect assessments with zero recognized delayed words
print("\n" + "=" * 70)
print("B. ZERO-WORD DELAYED-RECALL ASSESSMENTS")
print("=" * 70)

zero_word = best_three_way[
    best_three_way["STORY1DEL_NUM_WORDS"].eq(0)
].copy()

zero_columns = [
    "PTID",
    "RMT_Timepoint",
    "EXAMDATE_STORY",
    "DIAGNOSIS",
    "STORY1DEL_PLAYBACK_COUNT",
    "STORY1DEL_AUDIO_LENGTH",
    "STORY1DEL_NUM_WORDS",
    "STORY1DEL_SPEECH_DURATION",
    "STORY1DEL_NUM_PAUSES",
    "STORY1DEL_GMATCH",
    "STORY1DEL_GMATCH_IMM_DEL"
]

display(
    zero_word[[c for c in zero_columns if c in zero_word.columns]]
)


# C. Compare word counts across immediate and delayed responses
print("\n" + "=" * 70)
print("C. WORD COUNT DISTRIBUTIONS")
print("=" * 70)

word_columns = [
    "STORY1IMM_NUM_WORDS",
    "STORY2IMM_NUM_WORDS",
    "STORY1DEL_NUM_WORDS"
]

display(best_three_way[word_columns].describe().T)


# D. Inspect the six ECOG records excluded by the proposed quality rule
print("\n" + "=" * 70)
print("D. ECOG QUALITY EXCLUSIONS")
print("=" * 70)

excluded_ecog = best_three_way[
    best_three_way["VALIDITY"].ne(1)
    | best_three_way["VALIDITY"].isna()
]

display(
    excluded_ecog[
        [
            "PTID",
            "RMT_Timepoint",
            "EXAMDATE_STORY",
            "DIAGNOSIS",
            "VALIDITY",
            "STAFFASST",
            "ECOG_VALID_ITEMS"
        ]
    ]
)

# STEP 18: Identify zero-word responses across all Storyteller sections

sections = ["STORY1IMM", "STORY2IMM", "STORY1DEL"]

print("=" * 70)
print("A. ZERO-WORD RESPONSE COUNTS")
print("=" * 70)

zero_word_summary = []

for section in sections:
    word_col = f"{section}_NUM_WORDS"
    audio_col = f"{section}_AUDIO_LENGTH"
    duration_col = f"{section}_SPEECH_DURATION"
    playback_col = f"{section}_PLAYBACK_COUNT"

    zero_mask = best_three_way[word_col].eq(0)

    zero_word_summary.append({
        "Section": section,
        "Zero_word_assessments": int(zero_mask.sum()),
        "Missing_word_counts": int(best_three_way[word_col].isna().sum()),
        "Zero_audio_length": int(
            best_three_way.loc[zero_mask, audio_col].eq(0).sum()
        ),
        "Zero_speech_duration": int(
            best_three_way.loc[zero_mask, duration_col].eq(0).sum()
        )
    })

    print(f"\n{section}:")
    display(
        best_three_way.loc[
            zero_mask,
            [
                "PTID",
                "RMT_Timepoint",
                "EXAMDATE_STORY",
                "DIAGNOSIS",
                word_col,
                audio_col,
                duration_col,
                playback_col
            ]
        ]
    )

display(pd.DataFrame(zero_word_summary))


# B. Check zero-word responses among the proposed ECOG-quality cohort
print("\n" + "=" * 70)
print("B. ZERO-WORD RESPONSES IN ECOG-VALID ASSESSMENTS")
print("=" * 70)

ecog_valid_mask = best_three_way["VALIDITY"].eq(1)

for section in sections:
    word_col = f"{section}_NUM_WORDS"

    print(
        section,
        "| zero-word assessments among VALIDITY=1:",
        int(
            (
                ecog_valid_mask
                & best_three_way[word_col].eq(0)
            ).sum()
        )
    )


# C. Inspect all feature values for zero-word responses
print("\n" + "=" * 70)
print("C. ZERO-WORD RESPONSE FEATURE VALUES")
print("=" * 70)

for section in sections:
    word_col = f"{section}_NUM_WORDS"

    zero_rows = best_three_way[
        best_three_way[word_col].eq(0)
    ]

    if len(zero_rows) > 0:
        cols = [
            col for col in story_feature_columns
            if col.startswith(section)
        ]

        print(f"\n{section}")
        display(zero_rows[cols].T)

# STEP 19: Create Storyteller response-quality flags

quality_audit = best_three_way.copy()

# Identify zero-word responses in each section
for section in ["STORY1IMM", "STORY2IMM", "STORY1DEL"]:

    word_col = f"{section}_NUM_WORDS"
    duration_col = f"{section}_SPEECH_DURATION"

    quality_audit[f"{section}_ZERO_WORDS"] = (
        quality_audit[word_col].eq(0)
    )

    quality_audit[f"{section}_ZERO_SPEECH"] = (
        quality_audit[word_col].eq(0)
        & quality_audit[duration_col].eq(0)
    )

# Flag assessments with zero words in all three sections
zero_all_sections = (
    quality_audit[
        [
            "STORY1IMM_ZERO_WORDS",
            "STORY2IMM_ZERO_WORDS",
            "STORY1DEL_ZERO_WORDS"
        ]
    ].all(axis=1)
)

quality_audit["ZERO_WORDS_ALL_SECTIONS"] = zero_all_sections

# Flag missing delayed response
quality_audit["DELAYED_ALL_MISSING"] = (
    quality_audit[
        [
            col for col in story_feature_columns
            if col.startswith("STORY1DEL")
        ]
    ].isna().all(axis=1)
)

# A. Summary
print("=" * 70)
print("A. STORYTELLER RESPONSE QUALITY SUMMARY")
print("=" * 70)

print("Original assessments:", len(quality_audit))
print("ECOG-valid assessments:",
      int(quality_audit["VALIDITY"].eq(1).sum()))

print("Zero words in all sections:",
      int(quality_audit["ZERO_WORDS_ALL_SECTIONS"].sum()))

print("All delayed features missing:",
      int(quality_audit["DELAYED_ALL_MISSING"].sum()))

print("ECOG-valid assessments after excluding all-section zero-word cases:",
      int((
          quality_audit["VALIDITY"].eq(1)
          & ~quality_audit["ZERO_WORDS_ALL_SECTIONS"]
      ).sum()))

# B. Identify affected participants
print("\n" + "=" * 70)
print("B. AFFECTED ASSESSMENTS")
print("=" * 70)

display(
    quality_audit.loc[
        quality_audit["ZERO_WORDS_ALL_SECTIONS"]
        | quality_audit["DELAYED_ALL_MISSING"],
        [
            "PTID",
            "RMT_Timepoint",
            "EXAMDATE_STORY",
            "DIAGNOSIS",
            "VALIDITY",
            "ZERO_WORDS_ALL_SECTIONS",
            "DELAYED_ALL_MISSING"
        ]
    ]
)

# C. Cohort sizes under different quality rules
print("\n" + "=" * 70)
print("C. PROVISIONAL COHORT SIZES")
print("=" * 70)

rules = {
    "Original three-way cohort": pd.Series(
        True, index=quality_audit.index
    ),
    "ECOG validity = 1": quality_audit["VALIDITY"].eq(1),
    "ECOG valid + exclude all-section zero-word cases": (
        quality_audit["VALIDITY"].eq(1)
        & ~quality_audit["ZERO_WORDS_ALL_SECTIONS"]
    )
}

for name, mask in rules.items():
    subset = quality_audit.loc[mask]

    print(
        f"{name}: {len(subset)} assessments, "
        f"{subset['PTID'].nunique()} participants"
    )

    print("  CN:", int(subset["DIAGNOSIS"].eq(1).sum()))
    print("  MCI:", int(subset["DIAGNOSIS"].eq(2).sum()))



A. STORYTELLER / NOVOIC / RMT DICTIONARY SEARCH
Matching dictionary entries: 253


,PHASE,CRFNAME,TBLNAME,FLDNAME,TEXT,TYPE,CODE,UNITS
16,ADNI1,ADAS-Cognitive Behavior,ADAS,COT4LIST,Check each word correctly recalled.,T,0=Butter; 1=Arm; 2=Shore; 3=Letter; 4=Queen; 5=Cabin; 6=Pole; 7=Ticket; 8=Grass; 9=Engine;,NaN
80,ADNI1,ADAS-Cognitive Behavior Total Score,ADAS,TOTAL13,85 point total includingQ4 (Delayed Word Recall) and Q14 (Number Cancellation).,N,NaN,NaN
81,ADNI1,ADAS-Cognitive Behavior Total Score,ADAS,TOTSCORE,Classic 70 point total. Excludes Q4 (Delayed Word Recall) and Q14 (Number Cancellation).,N,NaN,NaN
93,ADNI1,ADAS-Cognitive Behavior,ADAS,Q1SCORE,ADAS Score Component Q1 Word Recall,N,0..10,NaN
100,ADNI1,ADAS-Cognitive Behavior,ADAS,Q1UNABLE,Task 1. Word Recall,N,0=Conducted;1=Not done (for reasons other than physical/cognitive);2=Participant refused;4=Participant unable to complete 3 trials for cognitive reasons,NaN
...,...,...,...,...,...,...,...,...
39357,NaN,Item Level Data,ITEM,ADAS_Q1_T1_WL3_W1_coast,"ADAS: ADAS Q1, T1-T3 indicate Trial 1 to Trial 3 of the Word-recall task. WL1-WL3 - indicates which wordlist was used (dependent on timepoint). W1-W10 followed by the word indicates the actual word and its position in the list.",T,0 = No; 1 = Yes; NULL = N/A,NaN
39358,NaN,Item Level Data,ITEM,ADAS_Q1_T1_WL3_W2_doll,"ADAS: ADAS Q1, T1-T3 indicate Trial 1 to Trial 3 of the Word-recall task. WL1-WL3 - indicates which wordlist was used (dependent on timepoint). W1-W10 followed by the word indicates the actual word and its position in the list.",T,0 = No; 1 = Yes; NULL = N/A,NaN
39359,NaN,Item Level Data,ITEM,ADAS_Q1_T1_WL3_W3_lip,"ADAS: ADAS Q1, T1-T3 indicate Trial 1 to Trial 3 of the Word-recall task. WL1-WL3 - indicates which wordlist was used (dependent on timepoint). W1-W10 followed by the word indicates the actual word and its position in the list.",T,0 = No; 1 = Yes; NULL = N/A,NaN
39360,NaN,Item Level Data,ITEM,ADAS_Q1_T1_WL3_W4_chair,"ADAS: ADAS Q1, T1-T3 indicate Trial 1 to Trial 3 of the Word-recall task. WL1-WL3 - indicates which wordlist was used (dependent on timepoint). W1-W10 followed by the word indicates the actual word and its position in the list.",T,0 = No; 1 = Yes; NULL = N/A,NaN



B. ECOG VALIDITY VS RESPONSE COMPLETENESS


ECOG_VALID_ITEMS,0,9,10,11,12,All
VALIDITY,,,,,,
1.0,0,0,0,8,137,145
2.0,0,1,1,1,0,3
NaN,3,0,0,0,0,3
All,3,1,1,9,137,151



C. INCOMPLETE DELAYED-RECALL ASSESSMENTS


,1499,1256,345,661,701
PTID,082_S_10840,099_S_10804,098_S_10358,135_S_5113,099_S_6396
RMT_Timepoint,m00,m00,m12,m00,m00
EXAMDATE_STORY,2025-08-14 00:00:00,2025-07-13 00:00:00,2025-02-17 00:00:00,2024-08-04 00:00:00,2024-08-27 00:00:00
DIAGNOSIS,2.0,2.0,1.0,1.0,1.0
VALIDITY,2.0,1.0,1.0,1.0,1.0
ECOG_VALID_ITEMS,9,12,12,12,12
STORY1DEL_LANGUAGE,en-US,en-US,en-US,en-US,NaN
STORY1DEL_PLAYBACK_COUNT,0.0,0.0,0.0,1.0,NaN
STORY1DEL_AUDIO_LENGTH,4.3,9.16,63.78,25.0,NaN
STORY1DEL_NUM_PRONOUNS,0.0,0.0,0.0,0.0,NaN


... (32 outputs left out)

,1499,1256,345,661
STORY1DEL_LANGUAGE,en-US,en-US,en-US,en-US
STORY1DEL_PLAYBACK_COUNT,0.0,0.0,0.0,1.0
STORY1DEL_AUDIO_LENGTH,4.3,9.16,63.78,25.0
STORY1DEL_NUM_PRONOUNS,0.0,0.0,0.0,0.0
STORY1DEL_NUM_NOUNS,0.0,0.0,0.0,0.0
STORY1DEL_NUM_PROPER_NOUNS,0.0,0.0,0.0,0.0
STORY1DEL_NUM_VERBS,0.0,0.0,0.0,0.0
STORY1DEL_NUM_ADVERBS,0.0,0.0,0.0,0.0
STORY1DEL_NUM_ADJECTIVES,0.0,0.0,0.0,0.0
STORY1DEL_NUM_WORDS,0.0,0.0,0.0,0.0


A. STORYTELLER RESPONSE QUALITY SUMMARY
Original assessments: 151
ECOG-valid assessments: 145
Zero words in all sections: 2
All delayed features missing: 1
ECOG-valid assessments after excluding all-section zero-word cases: 143

B. AFFECTED ASSESSMENTS


,PTID,RMT_Timepoint,EXAMDATE_STORY,DIAGNOSIS,VALIDITY,ZERO_WORDS_ALL_SECTIONS,DELAYED_ALL_MISSING
345,098_S_10358,m12,2025-02-17,1.0,1.0,True,False
661,135_S_5113,m00,2024-08-04,1.0,1.0,True,False
701,099_S_6396,m00,2024-08-27,1.0,1.0,False,True



C. PROVISIONAL COHORT SIZES
Original three-way cohort: 151 assessments, 141 participants
  CN: 105
  MCI: 46
ECOG validity = 1: 145 assessments, 135 participants
  CN: 102
  MCI: 43
ECOG valid + exclude all-section zero-word cases: 143 assessments, 133 participants
  CN: 100
  MCI: 43


## Primary cohort and candidate features

Define the primary analysis cohort and audit candidate predictive features.


In [53]:
# STEP 20: Define the primary analysis cohort

primary_mask = (
    quality_audit["VALIDITY"].eq(1)
    & ~quality_audit["ZERO_WORDS_ALL_SECTIONS"]
)

primary_cohort = quality_audit.loc[primary_mask].copy()

print("=" * 70)
print("PRIMARY ANALYSIS COHORT")
print("=" * 70)

print("Assessment records:", len(primary_cohort))
print("Unique participants:", primary_cohort["PTID"].nunique())

print("\nAssessment-level diagnosis:")
display(primary_cohort["DIAGNOSIS"].value_counts().rename(
    index={1: "CN", 2: "MCI"}
))

# Participant-level diagnosis
participant_labels = (
    primary_cohort.groupby("PTID")["DIAGNOSIS"]
    .agg(["nunique", "first", "count"])
)

print("\nParticipant-level diagnosis:")
display(
    participant_labels["first"]
    .value_counts()
    .rename(index={1: "CN", 2: "MCI"})
)

print("\nParticipants with repeated assessments:")
display(
    participant_labels["count"].value_counts().sort_index()
)

print(
    "\nParticipants with inconsistent diagnosis labels:",
    int((participant_labels["nunique"] > 1).sum())
)

# Check repeated assessments by diagnosis
print("\nRepeated assessment distribution by diagnosis:")
display(
    primary_cohort.groupby("DIAGNOSIS")["PTID"]
    .nunique()
    .rename(index={1: "CN", 2: "MCI"})
)

# Confirm delayed-recall missingness remains visible
delayed_cols = [
    c for c in primary_cohort.columns
    if c.startswith("STORY1DEL_")
    and pd.api.types.is_numeric_dtype(primary_cohort[c])
]

print("\nDelayed-recall feature missingness:")
display(
    (primary_cohort[delayed_cols].isna().mean() * 100)
    .round(2)
    .sort_values(ascending=False)
)

# STEP 21: Audit candidate predictive features

# Identify numeric Storyteller variables from the three sections
story_sections = ["STORY1IMM", "STORY2IMM", "STORY1DEL"]

story_numeric_cols = [
    col for col in primary_cohort.columns
    if any(col.startswith(section + "_") for section in story_sections)
    and pd.api.types.is_numeric_dtype(primary_cohort[col])
]

# Identify ECOG questionnaire items
ecog_item_cols = [
    f"ECOGPT{i}" for i in range(1, 13)
    if f"ECOGPT{i}" in primary_cohort.columns
]

print("Storyteller numeric columns:", len(story_numeric_cols))
print("ECOG item columns:", len(ecog_item_cols))

# Exclude variables that are metadata or have undocumented definitions
provisional_exclusions = [
    "PLAYBACK_COUNT",
    "GMATCH",
    "GMATCH_IMM_DEL"
]

story_candidate_cols = [
    col for col in story_numeric_cols
    if not any(term in col for term in provisional_exclusions)
]

print("Storyteller candidates after provisional exclusions:",
      len(story_candidate_cols))

# Build feature audit table
audit_cols = story_candidate_cols + ecog_item_cols

feature_audit = pd.DataFrame({
    "Feature": audit_cols,
    "Missing_Count": [
        primary_cohort[col].isna().sum() for col in audit_cols
    ],
    "Missing_Percent": [
        primary_cohort[col].isna().mean() * 100 for col in audit_cols
    ],
    "Unique_Values": [
        primary_cohort[col].nunique(dropna=True) for col in audit_cols
    ],
    "Constant": [
        primary_cohort[col].nunique(dropna=True) <= 1
        for col in audit_cols
    ]
})

feature_audit = feature_audit.sort_values(
    ["Constant", "Missing_Percent"],
    ascending=[False, False]
)

display(feature_audit.round(2))

print("\nConstant features:")
display(feature_audit.loc[feature_audit["Constant"]])

print("\nFeatures with missing values:")
display(
    feature_audit.loc[
        feature_audit["Missing_Count"] > 0
    ].sort_values("Missing_Percent", ascending=False)
)



PRIMARY ANALYSIS COHORT
Assessment records: 143
Unique participants: 133

Assessment-level diagnosis:


DIAGNOSIS
CN     100
MCI     43
Name: count, dtype: int64


Participant-level diagnosis:


first
CN     94
MCI    39
Name: count, dtype: int64


Participants with repeated assessments:


count
1    123
2     10
Name: count, dtype: int64

... (6 outputs left out)


Constant features:


,Feature,Missing_Count,Missing_Percent,Unique_Values,Constant
51,STORY1IMM_ZERO_WORDS,0,0.0,1,True
52,STORY1IMM_ZERO_SPEECH,0,0.0,1,True
53,STORY2IMM_ZERO_WORDS,0,0.0,1,True
54,STORY2IMM_ZERO_SPEECH,0,0.0,1,True



Features with missing values:


,Feature,Missing_Count,Missing_Percent,Unique_Values,Constant
61,ECOGPT5,4,2.797203,4,False
42,STORY1DEL_IDEA_DENSITY,2,1.398601,122,False
43,STORY1DEL_UNIQUE_WORDS_RATIO,2,1.398601,125,False
44,STORY1DEL_MEAN_LETTER_PER_WORD,2,1.398601,135,False
46,STORY1DEL_NONSPEECH_PROPORTION,2,1.398601,134,False
47,STORY1DEL_SPEECH_RATE,2,1.398601,135,False
48,STORY1DEL_ARTICULATION_RATE,2,1.398601,139,False
50,STORY1DEL_MEAN_PAUSE_LENGTH,2,1.398601,123,False
63,ECOGPT7,2,1.398601,4,False
34,STORY1DEL_AUDIO_LENGTH,1,0.699301,135,False


## Label, timing, and participant-overlap audits

Audit target coverage, temporal validity, and overlap among the existing dataframes.


In [54]:
# STEP 22: Inspect existing labelled and matched dataframes
# This does not reload files or modify existing data.

objects_to_check = [
    "story",
    "dx4",
    "ecog",
    "matched_ecog_dx",
    "latest_prior",
    "best_three_way",
    "primary_cohort"
]

for name in objects_to_check:

    obj = globals().get(name)

    print("\n" + "=" * 75)
    print(name.upper())

    if obj is None:
        print("Not available")
        continue

    print("Shape:", obj.shape)

    print("Relevant columns:")
    relevant = [
        col for col in obj.columns
        if any(term in str(col).upper() for term in [
            "PTID", "RID", "VISCODE", "EXAMDATE", "COLDATE",
            "DIAGNOSIS", "VALIDITY", "ECOGPT",
            "STORY1IMM", "STORY2IMM", "STORY1DEL",
            "RMT_TIMEPOINT", "TARGET"
        ])
    ]

    print(relevant)

# STEP 23: Audit diagnosis coverage and temporal validity
# No original dataframe is modified.

# ---------------------------------------------------------
# A. STORYTELLER LABEL AUDIT
# ---------------------------------------------------------

story_audit = latest_prior.copy()

story_audit["EXAMDATE_STORY"] = pd.to_datetime(
    story_audit["EXAMDATE_STORY"], errors="coerce"
)

story_audit["EXAMDATE_DX"] = pd.to_datetime(
    story_audit["EXAMDATE_DX"], errors="coerce"
)

# Keep only CN and MCI diagnoses
story_audit = story_audit[
    story_audit["DIAGNOSIS"].isin([1, 2])
].copy()

# Check diagnosis timing
story_audit["DX_AFTER_STORY"] = (
    story_audit["EXAMDATE_DX"] > story_audit["EXAMDATE_STORY"]
)

# Flag assessments with zero words in all three sections
story_zero_all = (
    story_audit["STORY1IMM_NUM_WORDS"].eq(0)
    & story_audit["STORY2IMM_NUM_WORDS"].eq(0)
    & story_audit["STORY1DEL_NUM_WORDS"].eq(0)
)

story_audit["ZERO_WORDS_ALL_SECTIONS"] = story_zero_all

# Provisional eligible Storyteller dataset
story_dataset_audit = story_audit.loc[
    ~story_audit["ZERO_WORDS_ALL_SECTIONS"]
    & ~story_audit["DX_AFTER_STORY"]
    & story_audit["EXAMDATE_DX"].notna()
    & story_audit["EXAMDATE_STORY"].notna()
].copy()

print("=" * 70)
print("A. STORYTELLER DATASET AUDIT")
print("=" * 70)

print("Rows with CN/MCI diagnosis:", len(story_audit))
print("Missing Storyteller dates:", story_audit["EXAMDATE_STORY"].isna().sum())
print("Missing diagnosis dates:", story_audit["EXAMDATE_DX"].isna().sum())
print("Diagnosis after Storyteller date:",
      story_audit["DX_AFTER_STORY"].sum())
print("Zero-word assessments in all sections:", story_zero_all.sum())

print("\nProvisional eligible Storyteller assessments:",
      len(story_dataset_audit))
print("Unique participants:",
      story_dataset_audit["PTID"].nunique())

print("\nDiagnosis distribution:")
display(
    story_dataset_audit["DIAGNOSIS"]
    .value_counts()
    .rename(index={1: "CN", 2: "MCI"})
)

# Check duplicate assessment keys
story_key_duplicates = story_dataset_audit.duplicated(
    ["PTID", "RMT_Timepoint", "EXAMDATE_STORY"]
).sum()

print("Duplicate participant-assessment keys:", story_key_duplicates)


# ---------------------------------------------------------
# B. ECOG LABEL AND TIMING AUDIT
# ---------------------------------------------------------

ecog_audit = matched_ecog_dx.copy()

ecog_audit["COLDATE"] = pd.to_datetime(
    ecog_audit["COLDATE"], errors="coerce"
)

ecog_audit["EXAMDATE"] = pd.to_datetime(
    ecog_audit["EXAMDATE"], errors="coerce"
)

# Convert ECOG item values to numeric and treat 9 as missing
ecog_items = [f"ECOGPT{i}" for i in range(1, 13)]

for col in ecog_items:
    ecog_audit[col] = pd.to_numeric(
        ecog_audit[col], errors="coerce"
    ).replace(9, np.nan)

ecog_audit["ECOG_VALID_ITEMS"] = ecog_audit[ecog_items].notna().sum(axis=1)

# Clinical diagnosis must be available on or before ECOG collection
ecog_audit["DX_AFTER_ECOG"] = (
    ecog_audit["EXAMDATE"] > ecog_audit["COLDATE"]
)

# Stage-by-stage eligibility counts
print("\n" + "=" * 70)
print("B. ECOG DATASET AUDIT")
print("=" * 70)

print("Exact participant + visit matches:", len(ecog_audit))
print("Unique participants:", ecog_audit["PTID"].nunique())

print("\nDiagnosis distribution before filtering:")
display(
    ecog_audit["DIAGNOSIS"]
    .value_counts(dropna=False)
    .rename(index={1: "CN", 2: "MCI", 3: "Dementia"})
)

print("\nECOG validity distribution:")
display(ecog_audit["VALIDITY"].value_counts(dropna=False))

print("\nValid ECOG item count distribution:")
display(ecog_audit["ECOG_VALID_ITEMS"].value_counts().sort_index())

print("\nMissing collection dates:", ecog_audit["COLDATE"].isna().sum())
print("Missing diagnosis dates:", ecog_audit["EXAMDATE"].isna().sum())
print("Diagnosis after ECOG collection:",
      ecog_audit["DX_AFTER_ECOG"].sum())

# Apply transparent eligibility criteria:
# - CN or MCI
# - VALIDITY = 1
# - At least 9 valid ECOG items
# - Diagnosis date on or before ECOG collection date
# - Both dates available

ecog_dataset_audit = ecog_audit.loc[
    ecog_audit["DIAGNOSIS"].isin([1, 2])
    & ecog_audit["VALIDITY"].eq(1)
    & ecog_audit["ECOG_VALID_ITEMS"].ge(9)
    & ~ecog_audit["DX_AFTER_ECOG"]
    & ecog_audit["COLDATE"].notna()
    & ecog_audit["EXAMDATE"].notna()
].copy()

print("\nProvisional eligible ECOG assessments:",
      len(ecog_dataset_audit))
print("Unique participants:",
      ecog_dataset_audit["PTID"].nunique())

print("\nDiagnosis distribution:")
display(
    ecog_dataset_audit["DIAGNOSIS"]
    .value_counts()
    .rename(index={1: "CN", 2: "MCI"})
)

# Check duplicate participant-visit keys
ecog_key_duplicates = ecog_dataset_audit.duplicated(
    ["PTID", "VISCODE"]
).sum()

print("Duplicate participant-visit keys:", ecog_key_duplicates)


# ---------------------------------------------------------
# C. PARTICIPANT OVERLAP AUDIT
# ---------------------------------------------------------

story_ids = set(story_dataset_audit["PTID"].dropna())
ecog_ids = set(ecog_dataset_audit["PTID"].dropna())

print("\n" + "=" * 70)
print("C. PARTICIPANT OVERLAP")
print("=" * 70)

print("Storyteller participants:", len(story_ids))
print("ECOG participants:", len(ecog_ids))
print("Shared participants:", len(story_ids & ecog_ids))
print("Storyteller-only participants:", len(story_ids - ecog_ids))
print("ECOG-only participants:", len(ecog_ids - story_ids))

# Compare against existing multimodal cohort
fusion_ids = set(primary_cohort["PTID"].dropna())

print("\nExisting fusion participants:", len(fusion_ids))
print("Fusion participants in Storyteller dataset:",
      len(fusion_ids & story_ids))
print("Fusion participants in ECOG dataset:",
      len(fusion_ids & ecog_ids))

print("\nAudit complete. No source datasets were modified.")

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# A. INSPECT THE 29 ALL-SECTION ZERO-WORD STORYTELLER ROWS
# ---------------------------------------------------------

story_audit_check = story_audit.copy()

word_cols = [
    "STORY1IMM_NUM_WORDS",
    "STORY2IMM_NUM_WORDS",
    "STORY1DEL_NUM_WORDS"
]

print("Word-count columns available:")
print([c for c in word_cols if c in story_audit_check.columns])

# Recalculate the flag without treating missing values as zero
for c in word_cols:
    story_audit_check[c] = pd.to_numeric(
        story_audit_check[c], errors="coerce"
    )

story_audit_check["ZERO_WORDS_RECALCULATED"] = (
    story_audit_check[word_cols].notna().all(axis=1)
    & story_audit_check[word_cols].eq(0).all(axis=1)
)

zero_rows = story_audit_check[
    story_audit_check["ZERO_WORDS_RECALCULATED"]
].copy()

print("\nRecalculated all-section zero-word assessments:", len(zero_rows))
print("Unique participants:", zero_rows["PTID"].nunique())

print("\nDiagnosis distribution:")
print(zero_rows["DIAGNOSIS"].value_counts(dropna=False))

display(
    zero_rows[
        ["PTID", "RMT_Timepoint", "EXAMDATE_STORY",
         "DIAGNOSIS", *word_cols]
    ].sort_values(["PTID", "EXAMDATE_STORY"])
)

print("\nMissing word counts in the three sections:")
print(story_audit_check[word_cols].isna().sum())

# ---------------------------------------------------------
# B. CHECK WHY FUSION PARTICIPANTS FAIL ECOG ELIGIBILITY
# ---------------------------------------------------------

fusion_check = primary_cohort.copy()

fusion_check["COLDATE"] = pd.to_datetime(
    fusion_check["COLDATE"], errors="coerce"
)
fusion_check["EXAMDATE_CLINICAL"] = pd.to_datetime(
    fusion_check["EXAMDATE_CLINICAL"], errors="coerce"
)

fusion_check["DX_AFTER_ECOG"] = (
    fusion_check["EXAMDATE_CLINICAL"] > fusion_check["COLDATE"]
)

print("\nFusion cohort diagnosis timing relative to ECOG:")
print(fusion_check["DX_AFTER_ECOG"].value_counts(dropna=False))

print("\nUnique fusion participants with diagnosis after ECOG:")
print(
    fusion_check.loc[
        fusion_check["DX_AFTER_ECOG"], "PTID"
    ].nunique()
)

print("\nFusion assessment-level diagnosis timing:")
display(
    fusion_check[
        ["PTID", "RMT_Timepoint", "COLDATE",
         "EXAMDATE_CLINICAL", "DIAGNOSIS",
         "DX_AFTER_ECOG"]
    ].sort_values(["PTID", "COLDATE"])
)

# ---------------------------------------------------------
# 1. ZERO-WORD ASSESSMENTS INSIDE THE CURRENT FUSION COHORT
# ---------------------------------------------------------

fusion_audit = primary_cohort.copy()

word_cols = [
    "STORY1IMM_NUM_WORDS",
    "STORY2IMM_NUM_WORDS",
    "STORY1DEL_NUM_WORDS"
]

for c in word_cols:
    fusion_audit[c] = pd.to_numeric(fusion_audit[c], errors="coerce")

fusion_audit["ZERO_ALL_SECTIONS"] = (
    fusion_audit[word_cols].notna().all(axis=1)
    & fusion_audit[word_cols].eq(0).all(axis=1)
)

print("Current fusion assessments:", len(fusion_audit))
print("Zero-word assessments in fusion:", fusion_audit["ZERO_ALL_SECTIONS"].sum())
print(
    "Participants with zero-word fusion assessments:",
    fusion_audit.loc[fusion_audit["ZERO_ALL_SECTIONS"], "PTID"].nunique()
)

# ---------------------------------------------------------
# 2. APPLY STRICT ECOG-DIAGNOSIS DATE RULE TO FUSION
# ---------------------------------------------------------

fusion_audit["COLDATE"] = pd.to_datetime(
    fusion_audit["COLDATE"], errors="coerce"
)
fusion_audit["EXAMDATE_CLINICAL"] = pd.to_datetime(
    fusion_audit["EXAMDATE_CLINICAL"], errors="coerce"
)

strict_fusion = fusion_audit[
    fusion_audit["COLDATE"].notna()
    & fusion_audit["EXAMDATE_CLINICAL"].notna()
    & (fusion_audit["EXAMDATE_CLINICAL"] <= fusion_audit["COLDATE"])
    & (~fusion_audit["ZERO_ALL_SECTIONS"])
].copy()

print("\nSTRICT TEMPORAL FUSION COHORT")
print("Assessments:", len(strict_fusion))
print("Participants:", strict_fusion["PTID"].nunique())
print("Diagnosis distribution:")
print(strict_fusion["DIAGNOSIS"].value_counts())
print("Participant diagnosis distribution:")
print(
    strict_fusion.groupby("PTID")["DIAGNOSIS"]
    .first()
    .value_counts()
)

# ---------------------------------------------------------
# 3. CHECK PARTICIPANT OVERLAP
# ---------------------------------------------------------

# ---------------------------------------------------------
# REBUILD ECOG ELIGIBILITY AND CHECK STRICT FUSION OVERLAP
# ---------------------------------------------------------

ecog_overlap_check = matched_ecog_dx.copy()

ecog_overlap_check["COLDATE"] = pd.to_datetime(
    ecog_overlap_check["COLDATE"], errors="coerce"
)
ecog_overlap_check["EXAMDATE"] = pd.to_datetime(
    ecog_overlap_check["EXAMDATE"], errors="coerce"
)

# Convert ECOG items to numeric and treat 9 as missing
ecog_items = [f"ECOGPT{i}" for i in range(1, 13)]

for col in ecog_items:
    ecog_overlap_check[col] = pd.to_numeric(
        ecog_overlap_check[col], errors="coerce"
    )
    ecog_overlap_check.loc[
        ecog_overlap_check[col] == 9, col
    ] = np.nan

ecog_overlap_check["ECOG_VALID_ITEMS"] = (
    ecog_overlap_check[ecog_items].notna().sum(axis=1)
)

# Apply strict eligibility conditions
ecog_overlap_check["ECOG_ELIGIBLE"] = (
    ecog_overlap_check["DIAGNOSIS"].isin([1, 2])
    & (ecog_overlap_check["VALIDITY"] == 1)
    & (ecog_overlap_check["ECOG_VALID_ITEMS"] >= 9)
    & ecog_overlap_check["COLDATE"].notna()
    & ecog_overlap_check["EXAMDATE"].notna()
    & (ecog_overlap_check["EXAMDATE"] <= ecog_overlap_check["COLDATE"])
)

eligible_ecog = ecog_overlap_check[
    ecog_overlap_check["ECOG_ELIGIBLE"]
].copy()

# Participant sets
strict_fusion_ids = set(strict_fusion["PTID"].dropna())
story_ids = set(story_audit.loc[
    story_audit["DIAGNOSIS"].isin([1, 2]), "PTID"
].dropna())
ecog_ids = set(eligible_ecog["PTID"].dropna())

print("STRICT FUSION PARTICIPANTS:", len(strict_fusion_ids))
print("ELIGIBLE STORYTELLER PARTICIPANTS:", len(story_ids))
print("ELIGIBLE ECOG PARTICIPANTS:", len(ecog_ids))

print("\nSTRICT FUSION OVERLAP")
print("Fusion participants in Storyteller:", len(strict_fusion_ids & story_ids))
print("Fusion participants in ECOG:", len(strict_fusion_ids & ecog_ids))

print("\nELIGIBLE ECOG DATASET")
print("Assessments:", len(eligible_ecog))
print("Participants:", eligible_ecog["PTID"].nunique())
print(eligible_ecog["DIAGNOSIS"].value_counts())

# Check whether any strict fusion participant is unexpectedly absent
missing_from_ecog = strict_fusion_ids - ecog_ids

print("\nFusion participants missing from eligible ECOG:", len(missing_from_ecog))
print(sorted(missing_from_ecog))

print("\nPARTICIPANT OVERLAP")
print("Strict fusion participants:", len(strict_fusion_ids))
print("In provisional Storyteller dataset:", len(strict_fusion_ids & story_ids))
print("In strictly eligible ECOG dataset:", len(strict_fusion_ids & ecog_ids))

# =========================================================



STORY
Shape: (15878, 73)
Relevant columns:
['PTID', 'RMT_Timepoint', 'EXAMDATE', 'STORY1IMM_LANGUAGE', 'STORY1IMM_PLAYBACK_COUNT', 'STORY1IMM_AUDIO_LENGTH', 'STORY1IMM_NUM_PRONOUNS', 'STORY1IMM_NUM_NOUNS', 'STORY1IMM_NUM_PROPER_NOUNS', 'STORY1IMM_NUM_VERBS', 'STORY1IMM_NUM_ADVERBS', 'STORY1IMM_NUM_ADJECTIVES', 'STORY1IMM_NUM_WORDS', 'STORY1IMM_IDEA_DENSITY', 'STORY1IMM_UNIQUE_WORDS_RATIO', 'STORY1IMM_MEAN_LETTER_PER_WORD', 'STORY1IMM_GMATCH', 'STORY1IMM_SPEECH_DURATION', 'STORY1IMM_NONSPEECH_PROPORTION', 'STORY1IMM_SPEECH_RATE', 'STORY1IMM_ARTICULATION_RATE', 'STORY1IMM_NUM_PAUSES', 'STORY1IMM_MEAN_PAUSE_LENGTH', 'STORY2IMM_LANGUAGE', 'STORY2IMM_PLAYBACK_COUNT', 'STORY2IMM_AUDIO_LENGTH', 'STORY2IMM_NUM_PRONOUNS', 'STORY2IMM_NUM_NOUNS', 'STORY2IMM_NUM_PROPER_NOUNS', 'STORY2IMM_NUM_VERBS', 'STORY2IMM_NUM_ADVERBS', 'STORY2IMM_NUM_ADJECTIVES', 'STORY2IMM_NUM_WORDS', 'STORY2IMM_IDEA_DENSITY', 'STORY2IMM_UNIQUE_WORDS_RATIO', 'STORY2IMM_MEAN_LETTER_PER_WORD', 'STORY2IMM_GMATCH', 'STORY2IMM_S

A. STORYTELLER DATASET AUDIT
Rows with CN/MCI diagnosis: 1459
Missing Storyteller dates: 0
Missing diagnosis dates: 0
Diagnosis after Storyteller date: 0
Zero-word assessments in all sections: 29

Provisional eligible Storyteller assessments: 1430
Unique participants: 490

Diagnosis distribution:


DIAGNOSIS
CN     1073
MCI     357
Name: count, dtype: int64

Duplicate participant-assessment keys: 0

B. ECOG DATASET AUDIT
Exact participant + visit matches: 1622
Unique participants: 1036

Diagnosis distribution before filtering:


DIAGNOSIS
CN          779
MCI         634
Dementia    207
NaN           2
Name: count, dtype: int64


ECOG validity distribution:


... (6 outputs left out)

,PTID,RMT_Timepoint,EXAMDATE_STORY,DIAGNOSIS,STORY1IMM_NUM_WORDS,STORY2IMM_NUM_WORDS,STORY1DEL_NUM_WORDS
328,006_S_6243,m00,2024-01-16,1.0,0.0,0.0,0.0
2302,006_S_6291,m18,2025-10-13,2.0,0.0,0.0,0.0
2852,019_S_10705,m18,2025-12-10,1.0,0.0,0.0,0.0
650,023_S_6547,m00,2024-04-19,1.0,0.0,0.0,0.0
653,023_S_6547,m06,2024-10-25,1.0,0.0,0.0,0.0
656,023_S_6547,m12,2025-04-22,1.0,0.0,0.0,0.0
2299,023_S_6547,m18,2025-10-22,1.0,0.0,0.0,0.0
3460,023_S_6547,m24,2026-04-21,1.0,0.0,0.0,0.0
3088,027_S_10494,m06,2025-11-11,1.0,0.0,0.0,0.0
3657,027_S_5083,m24,2026-06-07,1.0,0.0,0.0,0.0



Missing word counts in the three sections:
STORY1IMM_NUM_WORDS    1
STORY2IMM_NUM_WORDS    1
STORY1DEL_NUM_WORDS    2
dtype: int64

Fusion cohort diagnosis timing relative to ECOG:
DX_AFTER_ECOG
False    74
True     69
Name: count, dtype: int64

Unique fusion participants with diagnosis after ECOG:
65

Fusion assessment-level diagnosis timing:


,PTID,RMT_Timepoint,COLDATE,EXAMDATE_CLINICAL,DIAGNOSIS,DX_AFTER_ECOG
1528,002_S_4799,m00,2025-09-22,2025-09-22,1.0,False
943,002_S_6053,m00,2025-01-14,2025-01-14,1.0,False
1249,003_S_6996,m00,2025-05-15,2025-05-16,1.0,True
527,005_S_10288,m06,2024-11-07,2024-11-07,1.0,False
865,005_S_10302,m00,2024-12-12,2024-12-12,1.0,False
...,...,...,...,...,...,...
1193,305_S_6871,m12,2024-08-08,2024-09-04,2.0,True
722,941_S_10304,m06,2025-01-13,2025-01-13,1.0,False
438,941_S_6054,m00,2024-03-20,2024-03-20,1.0,False
499,941_S_6384,m00,2024-05-02,2024-05-02,1.0,False


Current fusion assessments: 143
Zero-word assessments in fusion: 0
Participants with zero-word fusion assessments: 0

STRICT TEMPORAL FUSION COHORT
Assessments: 74
Participants: 69
Diagnosis distribution:
DIAGNOSIS
1.0    51
2.0    23
Name: count, dtype: int64
Participant diagnosis distribution:
DIAGNOSIS
1.0    48
2.0    21
Name: count, dtype: int64
STRICT FUSION PARTICIPANTS: 69
ELIGIBLE STORYTELLER PARTICIPANTS: 495
ELIGIBLE ECOG PARTICIPANTS: 397

STRICT FUSION OVERLAP
Fusion participants in Storyteller: 69
Fusion participants in ECOG: 69

ELIGIBLE ECOG DATASET
Assessments: 501
Participants: 397
DIAGNOSIS
1.0    273
2.0    228
Name: count, dtype: int64

Fusion participants missing from eligible ECOG: 0
[]

PARTICIPANT OVERLAP
Strict fusion participants: 69
In provisional Storyteller dataset: 69
In strictly eligible ECOG dataset: 69


## Construct the source research datasets (Storyteller and ECog-12 PT)

Builds `dataset1_storyteller` and `dataset2_ecog` and defines the 51 Storyteller and 12 ECog-12 PT predictor columns. The primary 30-day matched multimodal cohort is built and saved in `03_train_temporal_and_ablation.ipynb`.


In [55]:
# STEP 24: CONSTRUCT THE THREE FINAL RESEARCH DATASETS
# =========================================================

# ---------------------------------------------------------
# A. DEFINE COMMON SETTINGS
# ---------------------------------------------------------

ecog_item_cols = [f"ECOGPT{i}" for i in range(1, 13)]

story_word_cols = [
    "STORY1IMM_NUM_WORDS",
    "STORY2IMM_NUM_WORDS",
    "STORY1DEL_NUM_WORDS"
]

# ---------------------------------------------------------
# B. DATASET 1: STORYTELLER ONLY
# ---------------------------------------------------------

dataset1_storyteller = latest_prior.copy()

# Parse dates
dataset1_storyteller["EXAMDATE_STORY"] = pd.to_datetime(
    dataset1_storyteller["EXAMDATE_STORY"], errors="coerce"
)

dataset1_storyteller["EXAMDATE_DX"] = pd.to_datetime(
    dataset1_storyteller["EXAMDATE_DX"], errors="coerce"
)

# Convert word counts to numeric
for col in story_word_cols:
    dataset1_storyteller[col] = pd.to_numeric(
        dataset1_storyteller[col], errors="coerce"
    )

# Identify assessments with exactly zero words in all sections.
# Missing word counts are not treated as zero.
dataset1_storyteller["ZERO_ALL_SECTIONS"] = (
    dataset1_storyteller[story_word_cols].notna().all(axis=1)
    & dataset1_storyteller[story_word_cols].eq(0).all(axis=1)
)

# Apply eligibility rules
dataset1_storyteller = dataset1_storyteller[
    dataset1_storyteller["DIAGNOSIS"].isin([1, 2])
    & dataset1_storyteller["PTID"].notna()
    & dataset1_storyteller["EXAMDATE_STORY"].notna()
    & dataset1_storyteller["EXAMDATE_DX"].notna()
    & (
        dataset1_storyteller["EXAMDATE_DX"]
        <= dataset1_storyteller["EXAMDATE_STORY"]
    )
    & (~dataset1_storyteller["ZERO_ALL_SECTIONS"])
].copy()

# Binary target: CN=0, MCI=1
dataset1_storyteller["TARGET"] = (
    dataset1_storyteller["DIAGNOSIS"].map({1: 0, 2: 1})
).astype(int)

# ---------------------------------------------------------
# C. DATASET 2: ECOG ONLY
# ---------------------------------------------------------

dataset2_ecog = eligible_ecog.copy()

# Ensure item values are numeric and code 9 is missing
for col in ecog_item_cols:
    dataset2_ecog[col] = pd.to_numeric(
        dataset2_ecog[col], errors="coerce"
    )
    dataset2_ecog.loc[
        dataset2_ecog[col] == 9, col
    ] = np.nan

# Reconfirm strict eligibility
dataset2_ecog["COLDATE"] = pd.to_datetime(
    dataset2_ecog["COLDATE"], errors="coerce"
)

dataset2_ecog["EXAMDATE"] = pd.to_datetime(
    dataset2_ecog["EXAMDATE"], errors="coerce"
)

dataset2_ecog = dataset2_ecog[
    dataset2_ecog["DIAGNOSIS"].isin([1, 2])
    & dataset2_ecog["PTID"].notna()
    & (dataset2_ecog["VALIDITY"] == 1)
    & (dataset2_ecog[ecog_item_cols].notna().sum(axis=1) >= 9)
    & dataset2_ecog["COLDATE"].notna()
    & dataset2_ecog["EXAMDATE"].notna()
    & (dataset2_ecog["EXAMDATE"] <= dataset2_ecog["COLDATE"])
].copy()

dataset2_ecog["TARGET"] = (
    dataset2_ecog["DIAGNOSIS"].map({1: 0, 2: 1})
).astype(int)

# ---------------------------------------------------------

# ---------------------------------------------------------
# D. DEFINE STORYTELLER PREDICTOR COLUMNS (51 FEATURES)
# ---------------------------------------------------------

story_candidates = [
    col for col in dataset1_storyteller.columns
    if col.startswith("STORY")
]

story_exclusions = [
    "PLAYBACK_COUNT",
    "GMATCH",
    "LANGUAGE"
]

story_feature_cols = [
    col for col in story_candidates
    if not any(term in col.upper() for term in story_exclusions)
]

for col in story_feature_cols:
    dataset1_storyteller[col] = pd.to_numeric(
        dataset1_storyteller[col], errors="coerce"
    )

story_feature_cols = [
    col for col in story_feature_cols
    if dataset1_storyteller[col].notna().any()
]

# ---------------------------------------------------------
# E. DEFINE PREDICTOR MATRICES
# ---------------------------------------------------------

X1_story = dataset1_storyteller[story_feature_cols].copy()
X2_ecog = dataset2_ecog[ecog_item_cols].copy()

y1_story = dataset1_storyteller["TARGET"].copy()
y2_ecog = dataset2_ecog["TARGET"].copy()

# ---------------------------------------------------------
# F. FINAL SOURCE DATASET SUMMARY
# ---------------------------------------------------------

def summarize_dataset(name, df):
    print(f"\n{name}")
    print("-" * 55)
    print("Assessments:", len(df))
    print("Participants:", df["PTID"].nunique())
    print("CN assessments:", (df["TARGET"] == 0).sum())
    print("MCI assessments:", (df["TARGET"] == 1).sum())
    print(
        "CN participants:",
        df.loc[df["TARGET"] == 0, "PTID"].nunique()
    )
    print(
        "MCI participants:",
        df.loc[df["TARGET"] == 1, "PTID"].nunique()
    )
    print("Duplicate rows by participant and assessment key:",
          df.duplicated(
              subset=["PTID", "RMT_Timepoint", "EXAMDATE_STORY"]
          ).sum()
          if "RMT_Timepoint" in df.columns
          else df.duplicated(subset=["PTID", "VISCODE"]).sum())

print("=" * 65)
print("FINAL SOURCE DATASET CONSTRUCTION SUMMARY")
print("=" * 65)

summarize_dataset("STORYTELLER SOURCE COHORT (dataset1_storyteller)", dataset1_storyteller)
summarize_dataset("ECOG-12 PT SOURCE COHORT (dataset2_ecog)", dataset2_ecog)

print("\nFEATURE SUMMARY")
print("-" * 55)
print("Storyteller features:", len(story_feature_cols))
print("ECog-12 PT features:", len(ecog_item_cols))

print("\nPREDICTOR MATRIX SHAPES")
print("X1_story:", X1_story.shape)
print("X2_ecog:", X2_ecog.shape)


FINAL SOURCE DATASET CONSTRUCTION SUMMARY

STORYTELLER SOURCE COHORT (dataset1_storyteller)
-------------------------------------------------------
Assessments: 1430
Participants: 490
CN assessments: 1073
MCI assessments: 357
CN participants: 361
MCI participants: 142
Duplicate rows by participant and assessment key: 0

ECOG-12 PT SOURCE COHORT (dataset2_ecog)
-------------------------------------------------------
Assessments: 501
Participants: 397
CN assessments: 273
MCI assessments: 228
CN participants: 240
MCI participants: 162
Duplicate rows by participant and assessment key: 0

FEATURE SUMMARY
-------------------------------------------------------
Storyteller features: 51
ECog-12 PT features: 12

PREDICTOR MATRIX SHAPES
X1_story: (1430, 51)
X2_ecog: (501, 12)


## Export the prepared source datasets to `data/processed/`


In [56]:
# EXPORT SOURCE DATASETS TO data/processed
# =========================================================

from pathlib import Path

current_path = Path.cwd().resolve()

project_root = next(
    (
        p for p in [current_path, *current_path.parents]
        if (p / "data" / "raw").is_dir() or (p / "data" / "processed").is_dir()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project folder containing data/raw or data/processed."
    )

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

dataset1_storyteller.to_csv(processed_dir / "dataset1_storyteller.csv", index=False)
dataset2_ecog.to_csv(processed_dir / "dataset2_ecog.csv", index=False)

print("Source datasets exported successfully.")
print("\nExport location:", processed_dir)

for filename in [
    "dataset1_storyteller.csv",
    "dataset2_ecog.csv",
]:
    file_path = processed_dir / filename
    print(f"{filename}: {file_path.stat().st_size / 1024:.2f} KB")


Source datasets exported successfully.

Export location: C:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\data\processed
dataset1_storyteller.csv: 1138.01 KB
dataset2_ecog.csv: 109.01 KB


## Dataset 4: all PTID-matched Storyteller × ECOG pairs

This section prepares the already-defined Dataset 4 structure: each eligible Storyteller assessment is paired with every eligible ECOG visit from the same participant. The ECOG visit’s matched DXSUM diagnosis defines the target: CN = 0 and MCI/dementia = 1. No 30-day restriction is applied between the two modalities.

The small compatibility aliases below connect this section to the variable names established earlier in this notebook. The preceding three-dataset code is unchanged.


In [57]:
# ============================================================
# DATASET 4 PREPARATION: ALL PTID-MATCHED STORYTELLER × ECOG PAIRS
#
# Target: CN (0) vs non-CN (MCI or dementia; 1)
#
# Each eligible Storyteller assessment is paired with every
# eligible ECOG visit for the same participant. The diagnosis
# attached to the ECOG visit supplies that pair's target.
#
# This cell uses separate d4_* variables and does not modify
# Datasets 1–3 or their exported files.
# ============================================================

# Map the Dataset 4 cell to names established in this notebook.
PROCESSED_DIR = processed_dir
ecog_feature_cols = ecog_item_cols

# ------------------------------------------------------------
# 1. Check that the source data and feature lists are available
# ------------------------------------------------------------

required_objects = [
    "dx",
    "ecog",
    "story",
    "RAW_DIR",
    "PROCESSED_DIR",
    "story_feature_cols",
    "ecog_feature_cols",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the three-dataset preparation cell first. "
        f"Missing variables: {missing_objects}"
    )

D4_OUTPUT_FILE = PROCESSED_DIR / "dataset4_multimodal_cn_vs_noncn.csv"

# Required Dataset 4 source fields.
required_d4_dx = [
    "PTID", "VISCODE", "EXAMDATE", "DIAGNOSIS", "PHASE"
]
required_d4_ecog = [
    "PTID", "VISCODE", "COLDATE", "VALIDITY",
    *ecog_feature_cols,
]
required_d4_story = [
    "PTID", "ADNIOnlineID", "RMT_Timepoint", "EXAMDATE",
    *story_feature_cols,
]

for name, df, required_columns in [
    ("DXSUM", dx, required_d4_dx),
    ("ECOG12PT", ecog, required_d4_ecog),
    ("STORYTELLER", story, required_d4_story),
]:
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing Dataset 4 columns: {missing}")

# ------------------------------------------------------------
# 2. Prepare separate copies of the raw sources
# ------------------------------------------------------------

d4_dx = dx.copy()
d4_ecog = ecog.copy()
d4_story = story.copy()

for df in (d4_dx, d4_ecog, d4_story):
    df["PTID"] = df["PTID"].astype("string").str.strip()

# Keep ADNI4 diagnoses and parse dates.
d4_dx = d4_dx.loc[
    d4_dx["PHASE"].astype("string").str.upper().eq("ADNI4")
].copy()

d4_dx["EXAMDATE"] = pd.to_datetime(
    d4_dx["EXAMDATE"], errors="coerce"
)
d4_ecog["COLDATE"] = pd.to_datetime(
    d4_ecog["COLDATE"], errors="coerce"
)
d4_story["EXAMDATE"] = pd.to_datetime(
    d4_story["EXAMDATE"], errors="coerce"
)

d4_dx["DIAGNOSIS"] = pd.to_numeric(
    d4_dx["DIAGNOSIS"], errors="coerce"
)
d4_ecog["VALIDITY"] = pd.to_numeric(
    d4_ecog["VALIDITY"], errors="coerce"
)

# Convert predictor values to numeric. ECOG code 9 is missing.
for col in story_feature_cols:
    d4_story[col] = pd.to_numeric(d4_story[col], errors="coerce")

for col in ecog_feature_cols:
    d4_ecog[col] = pd.to_numeric(
        d4_ecog[col], errors="coerce"
    ).replace(9, np.nan)

# ------------------------------------------------------------
# 3. Build eligible ECOG visits and attach visit diagnoses
# ------------------------------------------------------------

# Use the exact visit key, matching the Dataset 2 construction.
d4_ecog_for_merge = d4_ecog.dropna(
    subset=["PTID", "VISCODE", "COLDATE"]
).copy()

d4_dx_for_merge = d4_dx[
    ["PTID", "VISCODE", "EXAMDATE", "DIAGNOSIS"]
].dropna(
    subset=["PTID", "VISCODE", "EXAMDATE", "DIAGNOSIS"]
).copy()

# A visit must identify one ECOG record and one DXSUM diagnosis.
if d4_ecog_for_merge.duplicated(["PTID", "VISCODE"]).any():
    raise ValueError(
        "ECOG contains duplicate PTID + VISCODE keys; "
        "resolve these before constructing Dataset 4."
    )

if d4_dx_for_merge.duplicated(["PTID", "VISCODE"]).any():
    raise ValueError(
        "ADNI4 DXSUM contains duplicate PTID + VISCODE keys; "
        "resolve these before constructing Dataset 4."
    )

d4_ecog_dx = d4_ecog_for_merge.merge(
    d4_dx_for_merge,
    on=["PTID", "VISCODE"],
    how="inner",
    validate="one_to_one",
    suffixes=("_ECOG", "_DXSUM"),
)

# Retain CN, MCI, and dementia diagnosis codes.
# ADNI DXSUM codes used here: 1=CN, 2=MCI, 3=dementia.
d4_ecog_dx = d4_ecog_dx.loc[
    d4_ecog_dx["DIAGNOSIS"].isin([1, 2, 3])
    & d4_ecog_dx["VALIDITY"].eq(1)
    & (d4_ecog_dx["EXAMDATE"] <= d4_ecog_dx["COLDATE"])
].copy()

# ECOG requires at least 9 observed items; response code 9 is missing.
d4_ecog_dx["ECOG_VALID_ITEMS"] = (
    d4_ecog_dx[ecog_feature_cols].notna().sum(axis=1)
)

d4_ecog_dx = d4_ecog_dx.loc[
    d4_ecog_dx["ECOG_VALID_ITEMS"].ge(9)
].copy()

d4_ecog_dx["TARGET"] = (
    d4_ecog_dx["DIAGNOSIS"]
    .map({1: 0, 2: 1, 3: 1})
    .astype("int8")
)

# Give every retained ECOG visit a stable row identifier for the pair audit.
d4_ecog_dx = d4_ecog_dx.reset_index(drop=True)
d4_ecog_dx["_D4_ECOG_ROW_ID"] = np.arange(len(d4_ecog_dx))

# ------------------------------------------------------------
# 4. Prepare eligible Storyteller assessments
# ------------------------------------------------------------

# Keep dated Storyteller assessments with an assessment identifier.
# Apply the same zero-word quality exclusion used in the earlier
# Dataset 1 construction, without requiring a CN/MCI Story diagnosis.
d4_story = d4_story.dropna(
    subset=["PTID", "ADNIOnlineID", "RMT_Timepoint", "EXAMDATE"]
).copy()

d4_story["ZERO_WORDS_ALL_SECTIONS"] = (
    d4_story[
        [
            "STORY1IMM_NUM_WORDS",
            "STORY2IMM_NUM_WORDS",
            "STORY1DEL_NUM_WORDS",
        ]
    ].eq(0).all(axis=1)
)

d4_story = d4_story.loc[
    ~d4_story["ZERO_WORDS_ALL_SECTIONS"]
].copy()

# Preserve every eligible source row and identify it for the pair audit.
d4_story = d4_story.reset_index(drop=True)
d4_story["_D4_STORY_ROW_ID"] = np.arange(len(d4_story))

d4_story = d4_story.rename(
    columns={"EXAMDATE": "EXAMDATE_STORY"}
)

# ------------------------------------------------------------
# 5. Create all same-participant Storyteller × ECOG pairs
# ------------------------------------------------------------

# There is deliberately no 30-day date restriction between these
# two modalities. For each participant, form every Storyteller by
# every eligible ECOG-visit combination.
dataset4_multimodal_cn_vs_noncn = d4_story.merge(
    d4_ecog_dx,
    on="PTID",
    how="inner",
    suffixes=("_STORY", "_ECOG"),
)

if dataset4_multimodal_cn_vs_noncn.empty:
    raise ValueError(
        "No same-participant Storyteller × ECOG pairs were found."
    )

# Confirm each Storyteller/ECOG source-row combination appears once.
pair_key = ["_D4_STORY_ROW_ID", "_D4_ECOG_ROW_ID"]
duplicate_pair_count = int(
    dataset4_multimodal_cn_vs_noncn.duplicated(pair_key).sum()
)

if duplicate_pair_count:
    raise AssertionError(
        f"Found {duplicate_pair_count} duplicate Dataset 4 source pairs."
    )

# Verify the expected all-pairs count: sum of Storyteller × ECOG rows per PTID.
story_counts = d4_story.groupby("PTID").size()
ecog_counts = d4_ecog_dx.groupby("PTID").size()
shared_ptids = story_counts.index.intersection(ecog_counts.index)

expected_pair_count = int(
    sum(story_counts.loc[pid] * ecog_counts.loc[pid] for pid in shared_ptids)
)

if len(dataset4_multimodal_cn_vs_noncn) != expected_pair_count:
    raise AssertionError(
        "Dataset 4 row count does not match the expected "
        "Storyteller × ECOG all-pairs count."
    )

# ------------------------------------------------------------
# 6. Audit and export Dataset 4
# ------------------------------------------------------------

dataset4_summary = pd.DataFrame(
    [{
        "Dataset": "Dataset 4 - CN vs non-CN",
        "Target_definition": "CN=0; MCI/dementia=1",
        "Pairing": "All same-PTID Storyteller × eligible ECOG visits",
        "Rows": len(dataset4_multimodal_cn_vs_noncn),
        "Participants": dataset4_multimodal_cn_vs_noncn["PTID"].nunique(),
        "Storyteller_assessments": (
            dataset4_multimodal_cn_vs_noncn["_D4_STORY_ROW_ID"].nunique()
        ),
        "ECOG_visits": (
            dataset4_multimodal_cn_vs_noncn["_D4_ECOG_ROW_ID"].nunique()
        ),
        "TARGET_0_rows": int(
            (dataset4_multimodal_cn_vs_noncn["TARGET"] == 0).sum()
        ),
        "TARGET_1_rows": int(
            (dataset4_multimodal_cn_vs_noncn["TARGET"] == 1).sum()
        ),
        "Participants_with_changing_target": int(
            (
                dataset4_multimodal_cn_vs_noncn
                .groupby("PTID")["TARGET"]
                .nunique() > 1
            ).sum()
        ),
        "Storyteller_features": len(story_feature_cols),
        "ECOG_features": len(ecog_feature_cols),
        "Fusion_features": len(story_feature_cols) + len(ecog_feature_cols),
    }]
)

print("\nDATASET 4 SUMMARY")
display(dataset4_summary)

print("\nDataset 4 target counts:")
display(
    dataset4_multimodal_cn_vs_noncn["TARGET"]
    .value_counts()
    .sort_index()
    .rename(index={0: "CN", 1: "non-CN"})
    .to_frame("pair_rows")
)

print("\nDataset 4 pair audit")
print("Expected all-pairs rows:", expected_pair_count)
print("Constructed pair rows:", len(dataset4_multimodal_cn_vs_noncn))
print("Participants:", dataset4_multimodal_cn_vs_noncn["PTID"].nunique())
print(
    "Unique Storyteller assessments:",
    dataset4_multimodal_cn_vs_noncn["_D4_STORY_ROW_ID"].nunique()
)
print(
    "Unique eligible ECOG visits:",
    dataset4_multimodal_cn_vs_noncn["_D4_ECOG_ROW_ID"].nunique()
)
print("Duplicate source pairs:", duplicate_pair_count)

dataset4_multimodal_cn_vs_noncn.to_csv(
    D4_OUTPUT_FILE,
    index=False,
)

dataset4_summary.to_csv(
    PROCESSED_DIR / "dataset4_summary.csv",
    index=False,
)

print("\nDataset 4 exported to:")
print(D4_OUTPUT_FILE)



DATASET 4 SUMMARY


,Dataset,Target_definition,Pairing,Rows,Participants,Storyteller_assessments,ECOG_visits,TARGET_0_rows,TARGET_1_rows,Participants_with_changing_target,Storyteller_features,ECOG_features,Fusion_features
0,Dataset 4 - CN vs non-CN,CN=0; MCI/dementia=1,All same-PTID Storyteller × eligible ECOG visits,877,221,715,262,598,279,3,51,12,63



Dataset 4 target counts:


,pair_rows
TARGET,
CN,598
non-CN,279



Dataset 4 pair audit
Expected all-pairs rows: 877
Constructed pair rows: 877
Participants: 221
Unique Storyteller assessments: 715
Unique eligible ECOG visits: 262
Duplicate source pairs: 0

Dataset 4 exported to:
C:\Users\amnar\Documents\ADNI4_Storyteller_ECOG_Classification\data\processed\dataset4_multimodal_cn_vs_noncn.csv
